# Legacy Transformer Full Training - Standalone Replication

**Purpose:** Faithful replication of legacy training pipeline on full dataset with comprehensive metrics tracking compatible with the `moe_flashattn_4` experimentation framework.

**Bug fixes applied from refactored code:**
1. Removed `log_softmax` (correct for BCEWithLogitsLoss)
2. Fixed gradient clipping order (before optimizer.step)
3. Removed double weight update

**Data compatibility:**
- LOB embedding included (matching `moe_flashattn_4` input tensor: `[age, gender, lob, codes]`)
- Raw table column `target_cd` aliased as `target` for internal consistency
- Required columns: `individual_id`, `age_in_months`, `gender_cd`, `cd`, `dt_cnt`, `lob`, `target_cd`

**Metrics parity with `moe_flashattn_4`:**
- Batch-level: recall@K, precision@K, micro_recall@K, NDCG@20, positive_brier
- Epoch-level: LossTracker statistics (mean, std, min, max, improvement)
- Gradient tier analysis (common/medium/rare/tail gradient fraction)
- StreamingMetrics for validation (recall, precision, micro_recall, NDCG, MRR, Brier)
- Legacy metrics: train_loss, val_loss per epoch, training history

**Logs to:** `logs/{experiment_round}/legacy_replication/` (same structure as moe_flashattn_4)

**Data tables:**
- Full training: `a834793_Combined_All_LOB_o3_train_ending`
- 1.5M sample: `a834793_Combined_All_LOB_o3_train_10pct_sample`

**Reference files:**
- Original: `data_ingestion/Legacy/Train/python/min_transformer_train.py`
- Regenerated: `dev/transformer_training_pipeline.py`
- Cleaned: `dev/legacy/transformer_training_scoring.py`

In [20]:
import random
random.seed(1234)
import pandas as pd
import numpy as np
import gc
gc.collect()
import os
import time
import json
import logging
import torch
torch.manual_seed(123)
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import Dataset, DataLoader, random_split
from datetime import datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any, Set
import pytz
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

### Configuration

In [21]:
# ===========================================================================
# CONFIGURATION - Legacy values with corrections noted
# ===========================================================================
BIGQUERY_TABLE_FULL = 'edp-prod-storage.edp_ent_sdoheir_cns.a834793_Combined_All_LOB_o3_train_ending'
BIGQUERY_TABLE_10PCT = 'edp-prod-storage.edp_ent_sdoheir_cns.a834793_Combined_All_LOB_o3_train_10pct_sample'
GCP_PROJECT = 'edp-prod-css-sdoh'
# GCS_BUCKET = 'us-east4-edp-prod-css-sdoh--1b0f6fa9-bucket'
# MODEL_PATH = 'a834793_transformer/Model/legacy_replication'

batch_size = 512

embedding_size = 256
minimum_mth_training = 5      # days, not months
len_dy = 200                     # sequence length (days)
len_cd = 80                      # codes per day
nhead = 16                       # temporal encoder heads
nhid = 512                       # FFN hidden dim (legacy value, NOT 1024)
nlayers = 6                      # temporal encoder layers
ndropout = 0.05                  # dropout rate
cd_cnt = 75516                   # input vocabulary size
target_cd_cnt = 6297             # target vocabulary size
lob_vocab = 4                    # LOB categories (0=pad, 1=Commercial, 2=Medicare, 3=Medicaid)
parallel = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
entity_id = 'individual_id'
target = 'target'

NUM_EPOCHS = 1
VAL_SPLIT = 0.1
LEARNING_RATE = 1e-2             # Adjusted from legacy 1e-2 (double-update bug removed)
GRADIENT_CLIP = 0.25
LOG_INTERVAL = 100               # Batch metrics logging frequency
MICRO_BATCH_SIZE = 32            # Per-forward-pass batch size (fits in T4 GPU memory)
ACCUMULATION_STEPS = batch_size // MICRO_BATCH_SIZE  # = 16, effective batch_size=512
EXP_NAME = 'legacy_replication'
EXPERIMENT_ROUND = None          # Set before training, e.g. 'exp_round10_legacy'

print(f"Device: {device}")
print(f"GPUs available: {torch.cuda.device_count()}")
print(f"Config: batch={batch_size}, emb={embedding_size}, nhid={nhid}, "
      f"nhead={nhead}, nlayers={nlayers}, dropout={ndropout}")
print(f"Data: len_dy={len_dy}, len_cd={len_cd}, cd_cnt={cd_cnt}, "
      f"target_cd_cnt={target_cd_cnt}, lob_vocab={lob_vocab}")
print(f"Optimizer: SGD(lr={LEARNING_RATE}, momentum=0.9)")
print(f"Scheduler: CosineAnnealingLR(T_max={NUM_EPOCHS})")
print(f"Loss: BCEWithLogitsLoss (no pos_weight)")
print(f"Gradient clip: {GRADIENT_CLIP}")
print(f"Micro batch: {MICRO_BATCH_SIZE} x {ACCUMULATION_STEPS} = {MICRO_BATCH_SIZE * ACCUMULATION_STEPS} effective")

Device: cuda
GPUs available: 4
Config: batch=512, emb=256, nhid=512, nhead=16, nlayers=6, dropout=0.05
Data: len_dy=200, len_cd=80, cd_cnt=75516, target_cd_cnt=6297, lob_vocab=4
Optimizer: SGD(lr=0.01, momentum=0.9)
Scheduler: CosineAnnealingLR(T_max=1)
Loss: BCEWithLogitsLoss (no pos_weight)
Gradient clip: 0.25
Micro batch: 32 x 16 = 512 effective


## Logging Infrastructure
Replicates `MetricsLogger`, `LossTracker`, and `setup_experiment_logging` from `moe_flashattn_4.py` for full comparability.

In [40]:
def setup_experiment_logging(
    exp_name: str,
    log_dir: str = "logs",
    resume: bool = False
) -> logging.Logger:
    log_path = Path(log_dir) / exp_name
    log_path.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(exp_name)
    logger.setLevel(logging.DEBUG)
    logger.handlers = []

    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%H:%M:%S'
    )
    console_handler.setFormatter(console_formatter)
    logger.addHandler(console_handler)

    file_mode = 'a' if resume else 'w'
    file_handler = logging.FileHandler(log_path / 'training.log', mode=file_mode)
    file_handler.setLevel(logging.DEBUG)
    file_formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    file_handler.setFormatter(file_formatter)
    logger.addHandler(file_handler)

    if resume:
        logger.info(f"\n{'='*80}")
        logger.info(f"TRAINING RESUMED at {datetime.now()}")
        logger.info(f"{'='*80}\n")

    return logger


class MetricsLogger:
    """JSON-based metrics logger matching moe_flashattn_4 format exactly."""

    def __init__(self, exp_name: str, log_dir: str = "logs", resume: bool = False):
        self.exp_name = exp_name
        self.log_path = Path(log_dir) / exp_name
        self.log_path.mkdir(parents=True, exist_ok=True)
        self.epoch_metrics = []
        self.batch_metrics = []
        self.config = {}
        if resume:
            self._init_resume()

    def _init_resume(self):
        for fname, attr in [('epoch_metrics.json', 'epoch_metrics'),
                            ('batch_metrics.json', 'batch_metrics')]:
            fpath = self.log_path / fname
            if fpath.exists():
                try:
                    with open(fpath, 'r') as f:
                        setattr(self, attr, json.load(f))
                except Exception:
                    setattr(self, attr, [])

    def log_config(self, config: Dict):
        self.config = config

    def log_epoch(self, epoch: int, metrics: Dict[str, float]):
        self.epoch_metrics.append({'epoch': epoch, **metrics})

    def log_batch(self, epoch: int, batch: int, metrics: Dict[str, float]):
        self.batch_metrics.append({'epoch': epoch, 'batch': batch, **metrics})

    @staticmethod
    def convert_to_serializable(obj):
        if isinstance(obj, dict):
            return {k: MetricsLogger.convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [MetricsLogger.convert_to_serializable(item) for item in obj]
        elif isinstance(obj, (np.integer,)):
            return int(obj)
        elif isinstance(obj, (np.floating,)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif torch.is_tensor(obj):
            return obj.item() if obj.numel() == 1 else obj.cpu().tolist()
        elif isinstance(obj, (np.bool_, bool)):
            return bool(obj)
        elif isinstance(obj, torch.dtype):
            return str(obj)
        return obj

    def save(self):
        with open(self.log_path / 'epoch_metrics.json', 'w') as f:
            json.dump(self.convert_to_serializable(self.epoch_metrics), f, indent=2)
        with open(self.log_path / 'batch_metrics.json', 'w') as f:
            json.dump(self.convert_to_serializable(self.batch_metrics), f, indent=2)
        if self.config:
            with open(self.log_path / 'config.json', 'w') as f:
                json.dump(self.convert_to_serializable(self.config), f, indent=2)

    def save_final_results(self, results: Dict):
        results_path = self.log_path / 'final_results.json'
        with open(results_path, 'w') as f:
            json.dump(self.convert_to_serializable(results), f, indent=2)
        return results_path

    def get_summary(self) -> Dict:
        if not self.epoch_metrics:
            return {}
        real_epochs = [m for m in self.epoch_metrics if 'resume_event' not in m]
        return {
            'num_epochs': len(real_epochs),
            'best_val_loss': min((m.get('val_loss', float('inf')) for m in real_epochs), default=float('inf')),
            'final_train_loss': real_epochs[-1].get('train_loss', None) if real_epochs else None,
        }

# ---------------------------------------------------------------------------
# Early stopping: primary_metric and dependent parameters
# ---------------------------------------------------------------------------
# Available metrics (from StreamingMetrics / evaluate()):
#   - val_loss          : mean BCE over validation batches (set mode='min')
#   - ndcg@5, ndcg@10, ndcg@20 : normalized DCG at K (set mode='max'); ndcg@20 recommended for clinical ranking
#   - recall@5, recall@10, recall@20 : hit-at-K recall (set mode='max')
#   - micro_recall@5, micro_recall@10, micro_recall@20 : micro-averaged recall (set mode='max')
#   - mrr               : mean reciprocal rank (set mode='max')
#   - precision@5, precision@10, precision@20 (set mode='max')
# When choosing primary_metric:
#   - ndcg@20 / recall@K / mrr / precision@K  -> mode='max', min_delta positive improvement
#   - val_loss                                 -> mode='min', min_delta positive improvement
#   - If using val_loss: consider lower min_delta (e.g. 1e-4); patience may need tuning
#   - If using rank metrics: val_fraction and val_check_interval affect variance; 0.2 and 500 are reasonable
@dataclass
class EarlyStoppingConfig:
    """Configuration for sub-epoch early stopping with clinically-tailored metrics."""
    enabled: bool = True
    # primary_metric: one of val_loss, ndcg@5/10/20, recall@5/10/20, micro_recall@5/10/20, mrr, precision@5/10/20 (see comment block above)
    primary_metric: str = 'micro_recall@20'
    mode: str = 'max'                      # 'max' for NDCG/recall/MRR/precision; 'min' for val_loss
    patience: int = 5                      # validation checks without improvement before stopping
    min_delta: float = 0.001               # minimum improvement to count as "better"
    warmup_steps: int = 1000               # skip early stopping during initial optimizer steps
    val_check_interval: int = 500          # run validation every N optimizer steps
    val_fraction: float = 0.2               # fraction of val set for sub-epoch checks (1.0 at epoch end)
    train_loss_slope_window: int = 500     # optimizer steps window for plateau detection
    train_loss_slope_threshold: float = 1e-5  # minimum loss decrease rate per step
    restore_best: bool = True              # restore best checkpoint when stopping

    def __post_init__(self):
        assert self.mode in ('min', 'max'), f"mode must be 'min' or 'max', got '{self.mode}'"
        assert self.patience >= 1, "patience must be >= 1"
        assert 0.0 < self.val_fraction <= 1.0, "val_fraction must be in (0, 1]"
        assert self.val_check_interval >= 1, "val_check_interval must be >= 1"


class EarlyStoppingMonitor:
    """Sub-epoch early stopping monitor with clinically-tailored metric tracking.

    Integrates with the existing training loop to provide:
    - Configurable sub-epoch validation at optimizer-step intervals
    - Primary metric tracking (NDCG@20 by default) with patience
    - Warmup period to protect CosineAnnealingLR exploration phase
    - Train loss slope detection as a secondary signal
    - Best-model checkpoint tracking independent of stopping
    """

    def __init__(self, config: EarlyStoppingConfig, logger=None):
        self.config = config
        self.logger = logger

        self._best_metric = float('-inf') if config.mode == 'max' else float('inf')
        self._best_step = 0
        self._best_checkpoint_path = None
        self._checks_without_improvement = 0
        self._total_checks = 0
        self._should_stop = False

        self._val_history = []     # list of (global_step, metric_value, full_metrics_dict)
        self._train_loss_buffer = []  # list of (global_step, smoothed_loss)

    def _is_improvement(self, current: float) -> bool:
        if self.config.mode == 'max':
            return current > self._best_metric + self.config.min_delta
        return current < self._best_metric - self.config.min_delta

    def should_validate(self, global_step: int) -> bool:
        """Check if we should run validation at this optimizer step."""
        if not self.config.enabled:
            return False
        return global_step > 0 and global_step % self.config.val_check_interval == 0

    def record_validation(self, global_step: int, metrics: dict) -> dict:
        """Record a validation result and return a status dict.

        Returns dict with keys: improved, should_stop, metric_value,
        best_metric, checks_without_improvement, in_warmup.
        """
        metric_value = metrics.get(self.config.primary_metric, None)
        if metric_value is None:
            available = [k for k in metrics if 'ndcg' in k or 'recall' in k or 'loss' in k]
            raise KeyError(
                f"Primary metric '{self.config.primary_metric}' not in validation results. "
                f"Available metric-like keys: {available}"
            )

        self._val_history.append((global_step, metric_value, metrics))
        self._total_checks += 1

        in_warmup = global_step < self.config.warmup_steps
        improved = self._is_improvement(metric_value)

        if improved:
            self._best_metric = metric_value
            self._best_step = global_step
            self._checks_without_improvement = 0
        else:
            if not in_warmup:
                self._checks_without_improvement += 1

        if not in_warmup and self._checks_without_improvement >= self.config.patience:
            self._should_stop = True

        status = {
            'improved': improved,
            'should_stop': self._should_stop,
            'metric_value': metric_value,
            'best_metric': self._best_metric,
            'best_step': self._best_step,
            'checks_without_improvement': self._checks_without_improvement,
            'in_warmup': in_warmup,
            'total_checks': self._total_checks,
        }

        if self.logger:
            phase = "WARMUP" if in_warmup else "ACTIVE"
            marker = " ***NEW BEST***" if improved else ""
            self.logger.info(
                f"[EarlyStop|{phase}] step={global_step} "
                f"{self.config.primary_metric}={metric_value:.4f} "
                f"best={self._best_metric:.4f}@step{self._best_step} "
                f"patience={self._checks_without_improvement}/{self.config.patience}"
                f"{marker}"
            )

        return status

    def record_train_loss(self, global_step: int, smoothed_loss: float):
        """Record smoothed training loss for slope detection."""
        self._train_loss_buffer.append((global_step, smoothed_loss))

    def detect_train_loss_plateau(self) -> bool:
        """Check if training loss has plateaued using the configured window.

        Returns True if the loss slope over the last `train_loss_slope_window`
        optimizer steps is below the threshold.
        """
        window = self.config.train_loss_slope_window
        if len(self._train_loss_buffer) < window:
            return False

        recent = self._train_loss_buffer[-window:]
        first_loss = recent[0][1]
        last_loss = recent[-1][1]
        step_span = recent[-1][0] - recent[0][0]
        if step_span == 0:
            return False

        slope = (first_loss - last_loss) / step_span
        return slope < self.config.train_loss_slope_threshold

    @property
    def should_stop(self) -> bool:
        return self._should_stop

    @property
    def best_metric(self) -> float:
        return self._best_metric

    @property
    def best_step(self) -> int:
        return self._best_step

    @property
    def best_checkpoint_path(self) -> str:
        return self._best_checkpoint_path

    @best_checkpoint_path.setter
    def best_checkpoint_path(self, path: str):
        self._best_checkpoint_path = path

    def get_summary(self) -> dict:
        """Return a summary dict for logging/serialization."""
        return {
            'enabled': self.config.enabled,
            'primary_metric': self.config.primary_metric,
            'best_metric': self._best_metric,
            'best_step': self._best_step,
            'total_checks': self._total_checks,
            'stopped_early': self._should_stop,
            'checks_without_improvement': self._checks_without_improvement,
            'val_history': [
                {'step': s, self.config.primary_metric: v}
                for s, v, _ in self._val_history
            ],
        }

# Why training loss can be greater than validation loss:
# (1) Train uses dropout and model.train(); val uses model.eval() (no dropout), so train loss is often higher.
# (2) train_loss is mean over optimizer-step batch losses; val_loss is mean over val batches (different sample composition).
# (3) train_loss_last below is the loss at the last optimizer step (final loss at epoch end), for comparison with train_loss_mean.
class LossTracker:
    """Track training loss trajectory for learning curve analysis (matches moe_flashattn_4).
    Provides train_loss_mean (mean over epoch) and train_loss_last (final loss at epoch end)."""

    def __init__(self, window_size: int = 100):
        self.window_size = window_size
        self.reset_epoch()
        self.epoch_summaries = []

    def reset_epoch(self):
        self.batch_losses = []
        self.batch_steps = []
        self.running_sum = 0.0
        self.running_count = 0

    def log_batch(self, loss: float, step: int):
        self.batch_losses.append(loss)
        self.batch_steps.append(step)
        self.running_sum += loss
        self.running_count += 1

    def get_recent_losses(self, n: int = None) -> List[float]:
        if n is None:
            n = self.window_size
        return self.batch_losses[-n:] if len(self.batch_losses) >= n else self.batch_losses

    def get_epoch_summary(self) -> Dict[str, float]:
        if len(self.batch_losses) == 0:
            return {}
        losses_array = np.array(self.batch_losses)
        summary = {
            'train_loss_mean': float(np.mean(losses_array)),
            'train_loss_std': float(np.std(losses_array)),
            'train_loss_min': float(np.min(losses_array)),
            'train_loss_max': float(np.max(losses_array)),
            'train_loss_first': float(losses_array[0]),
            'train_loss_last': float(losses_array[-1]),
            'train_loss_improvement': float(losses_array[0] - losses_array[-1]),
        }
        if len(losses_array) >= 100:
            smoothed = np.convolve(losses_array, np.ones(100)/100, mode='valid')
            summary['train_loss_smoothed'] = float(smoothed[-1])
        self.epoch_summaries.append(summary)
        return summary

    def save_trajectory(self, filepath: str):
        trajectory = {
            'steps': self.batch_steps,
            'losses': self.batch_losses,
            'epoch_summaries': self.epoch_summaries
        }
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, 'w') as f:
            json.dump(trajectory, f, indent=2)

    def should_stop_early(self, patience: int = 3) -> bool:
        if len(self.epoch_summaries) < patience + 1:
            return False
        recent = [s['train_loss_mean'] for s in self.epoch_summaries[-patience:]]
        return all(recent[i] >= recent[i-1] for i in range(1, len(recent)))

print("Logging infrastructure loaded.")

Logging infrastructure loaded.


In [41]:
class GradientTierAnalyzer:
    """
    Analyzes gradient contribution per code frequency tier.
    Diagnoses if rare/tail codes receive insufficient gradient signal.
    Matches moe_flashattn_4 pattern: log_batch() -> aggregate_epoch() -> get_diagnosis()
    """

    def __init__(self, code_frequencies: np.ndarray, device: torch.device, log_interval: int = 500):
        self.device = device
        self.log_interval = log_interval
        self.num_codes = len(code_frequencies)

        freq_nz = code_frequencies[code_frequencies > 0]
        if len(freq_nz) == 0:
            raise ValueError("No non-zero frequencies found")

        percentiles = np.percentile(freq_nz, [20, 50, 80])

        self.tier_indices = {}
        self.tier_sizes = {}

        common_mask = code_frequencies > percentiles[2]
        self.tier_indices['common'] = torch.tensor(np.where(common_mask)[0], dtype=torch.long)
        self.tier_sizes['common'] = int(common_mask.sum())

        medium_mask = (code_frequencies <= percentiles[2]) & (code_frequencies > percentiles[1])
        self.tier_indices['medium'] = torch.tensor(np.where(medium_mask)[0], dtype=torch.long)
        self.tier_sizes['medium'] = int(medium_mask.sum())

        rare_mask = (code_frequencies <= percentiles[1]) & (code_frequencies > percentiles[0])
        self.tier_indices['rare'] = torch.tensor(np.where(rare_mask)[0], dtype=torch.long)
        self.tier_sizes['rare'] = int(rare_mask.sum())

        tail_mask = (code_frequencies <= percentiles[0]) & (code_frequencies > 0)
        self.tier_indices['tail'] = torch.tensor(np.where(tail_mask)[0], dtype=torch.long)
        self.tier_sizes['tail'] = int(tail_mask.sum())

        self.batch_buffer = []
        self.epoch_summaries = []

        print(f"  GradientTierAnalyzer initialized:")
        print(f"    Common: {self.tier_sizes['common']} codes")
        print(f"    Medium: {self.tier_sizes['medium']} codes")
        print(f"    Rare:   {self.tier_sizes['rare']} codes")
        print(f"    Tail:   {self.tier_sizes['tail']} codes")

    def _get_decoder_gradients(self, model: nn.Module) -> Optional[torch.Tensor]:
        actual_model = model
        if isinstance(model, nn.DataParallel):
            actual_model = model.module

        decoder = None
        if hasattr(actual_model, 'decoder_cd'):
            decoder = actual_model.decoder_cd
        else:
            for name, module in actual_model.named_modules():
                if 'decoder_cd' in name and isinstance(module, nn.Linear):
                    decoder = module
                    break

        if decoder is None or decoder.weight.grad is None:
            return None
        return decoder.weight.grad.detach()

    def log_batch(self, model: nn.Module, batch_idx: int) -> Dict[str, float]:
        if batch_idx % self.log_interval != 0:
            return {}

        grad = self._get_decoder_gradients(model)
        if grad is None:
            return {}

        grad_cpu = grad.cpu()
        per_code_norm = torch.norm(grad_cpu, dim=1)
        total_norm = per_code_norm.sum().item()

        if total_norm < 1e-12:
            return {}

        metrics = {}
        for tier_name, indices in self.tier_indices.items():
            if len(indices) == 0:
                metrics[f'grad_tier_{tier_name}_frac'] = 0.0
                metrics[f'grad_tier_{tier_name}_norm'] = 0.0
                continue
            tier_norms = per_code_norm[indices]
            tier_total = tier_norms.sum().item()
            metrics[f'grad_tier_{tier_name}_frac'] = tier_total / total_norm
            metrics[f'grad_tier_{tier_name}_norm'] = tier_norms.mean().item()

        metrics['grad_tier_total_norm'] = total_norm
        self.batch_buffer.append(metrics)
        return metrics

    def aggregate_epoch(self) -> Dict[str, float]:
        if not self.batch_buffer:
            return {}
        epoch_summary = {'train_grad_tier_samples': len(self.batch_buffer)}
        for key in self.batch_buffer[0].keys():
            values = [m[key] for m in self.batch_buffer]
            epoch_summary[f'train_{key}'] = np.mean(values)
            epoch_summary[f'train_{key}_std'] = np.std(values)
        self.epoch_summaries.append(epoch_summary)
        self.batch_buffer = []
        return epoch_summary

    def get_diagnosis(self) -> Dict[str, Any]:
        if not self.epoch_summaries:
            return {}
        latest = self.epoch_summaries[-1]
        common_frac = latest.get('train_grad_tier_common_frac', 0.0)
        tail_frac = latest.get('train_grad_tier_tail_frac', 0.0)
        return {
            'tier_sizes': self.tier_sizes,
            'final_epoch_summary': latest,
            'all_epoch_summaries': self.epoch_summaries,
            'gradient_imbalance_ratio': common_frac / max(tail_frac, 1e-8),
        }

print("GradientTierAnalyzer loaded.")

GradientTierAnalyzer loaded.


In [42]:
@dataclass
class StreamingMetricsState:
    """Internal state for streaming metrics computation."""
    total_loss: float = 0.0
    num_batches: int = 0
    num_samples: int = 0
    recall_hits: Dict[int, int] = field(default_factory=dict)
    recall_total: Dict[int, int] = field(default_factory=dict)
    micro_recall_hits: Dict[int, int] = field(default_factory=dict)
    micro_recall_true: Dict[int, int] = field(default_factory=dict)
    precision_sum: Dict[int, float] = field(default_factory=dict)
    precision_count: Dict[int, int] = field(default_factory=dict)
    ndcg_sum: Dict[int, float] = field(default_factory=dict)
    ndcg_count: Dict[int, int] = field(default_factory=dict)
    mrr_sum: float = 0.0
    mrr_count: int = 0
    positive_brier_sum: float = 0.0
    positive_brier_count: int = 0


class StreamingMetrics:
    """Memory-efficient streaming metrics aggregator matching moe_flashattn_4."""

    def __init__(self, k_values: Tuple[int, ...] = (1, 5, 10, 20),
                 compute_mrr: bool = True, compute_brier: bool = True,
                 vocab_size: int = 6297):
        self.k_values = k_values
        self.compute_mrr = compute_mrr
        self.compute_brier = compute_brier
        self.vocab_size = vocab_size
        self._max_k = max(k_values)
        self._discounts = 1.0 / np.log2(np.arange(2, self._max_k + 2))
        self._discount_cumsum = np.cumsum(self._discounts)
        self._cached_device = None
        self._discounts_tensor = None
        self._discount_cumsum_tensor = None
        self.reset()

    def reset(self):
        self._state = StreamingMetricsState(
            recall_hits={k: 0 for k in self.k_values},
            recall_total={k: 0 for k in self.k_values},
            micro_recall_hits={k: 0 for k in self.k_values},
            micro_recall_true={k: 0 for k in self.k_values},
            precision_sum={k: 0.0 for k in self.k_values},
            precision_count={k: 0 for k in self.k_values},
            ndcg_sum={k: 0.0 for k in self.k_values},
            ndcg_count={k: 0 for k in self.k_values},
        )

    def _get_tensors_for_device(self, device: torch.device):
        if self._cached_device != device:
            self._discounts_tensor = torch.tensor(self._discounts, dtype=torch.float32, device=device)
            self._discount_cumsum_tensor = torch.tensor(self._discount_cumsum, dtype=torch.float32, device=device)
            self._cached_device = device
        return self._discounts_tensor, self._discount_cumsum_tensor

    def update_loss(self, loss: float):
        self._state.total_loss += loss
        self._state.num_batches += 1

    def update(self, predictions: torch.Tensor, targets: List[List[int]]):
        batch_size_local = predictions.shape[0]
        dev = predictions.device
        discounts_tensor, discount_cumsum_tensor = self._get_tensors_for_device(dev)

        with torch.no_grad():
            _, top_k_indices = torch.topk(predictions, self._max_k, dim=-1)

        target_tensor = torch.zeros(batch_size_local, self.vocab_size, dtype=torch.bool, device=dev)
        valid_mask = torch.zeros(batch_size_local, dtype=torch.bool, device=dev)
        num_true_per_sample = torch.zeros(batch_size_local, dtype=torch.long, device=dev)

        for i, target_codes in enumerate(targets):
            valid_codes = [c for c in target_codes if 0 < c < self.vocab_size]
            if valid_codes:
                target_tensor[i, valid_codes] = True
                valid_mask[i] = True
                num_true_per_sample[i] = len(valid_codes)

        num_valid = valid_mask.sum().item()
        if num_valid == 0:
            return
        self._state.num_samples += num_valid

        for k in self.k_values:
            top_k = top_k_indices[:, :k]
            hits_matrix = torch.gather(target_tensor, 1, top_k)
            hits_per_sample = hits_matrix.sum(dim=1)
            valid_hits = hits_per_sample[valid_mask]
            valid_num_true = num_true_per_sample[valid_mask]

            self._state.recall_hits[k] += (valid_hits > 0).sum().item()
            self._state.recall_total[k] += num_valid
            self._state.micro_recall_hits[k] += valid_hits.sum().item()
            self._state.micro_recall_true[k] += valid_num_true.sum().item()
            self._state.precision_sum[k] += (valid_hits.float() / k).sum().item()
            self._state.precision_count[k] += num_valid

        for k in self.k_values:
            top_k = top_k_indices[:, :k]
            hits_matrix = torch.gather(target_tensor, 1, top_k).float()
            dcg_per_sample = (hits_matrix * discounts_tensor[:k]).sum(dim=1)
            valid_num_true_k = torch.clamp(num_true_per_sample, max=k)
            idcg_indices = (valid_num_true_k - 1).clamp(min=0)
            idcg_per_sample = torch.where(
                valid_num_true_k > 0,
                discount_cumsum_tensor[idcg_indices],
                torch.zeros(batch_size_local, device=dev)
            )
            ndcg_per_sample = torch.where(
                idcg_per_sample > 0,
                dcg_per_sample / idcg_per_sample,
                torch.zeros_like(dcg_per_sample)
            )
            self._state.ndcg_sum[k] += ndcg_per_sample[valid_mask].sum().item()
            self._state.ndcg_count[k] += num_valid

        if self.compute_mrr:
            hits_matrix = torch.gather(target_tensor, 1, top_k_indices).float()
            ranks = torch.arange(1, self._max_k + 1, device=dev, dtype=torch.float32)
            first_hit_mask = (hits_matrix.cumsum(dim=1) == 1) & (hits_matrix == 1)
            rr = (first_hit_mask.float() / ranks).sum(dim=1)
            self._state.mrr_sum += rr[valid_mask].sum().item()
            self._state.mrr_count += num_valid

        if self.compute_brier:
            probs_tensor = torch.sigmoid(predictions)
            for i in range(batch_size_local):
                if valid_mask[i]:
                    pos_indices = target_tensor[i].nonzero(as_tuple=True)[0]
                    if len(pos_indices) > 0:
                        pos_probs = probs_tensor[i, pos_indices]
                        self._state.positive_brier_sum += ((pos_probs - 1.0) ** 2).sum().item()
                        self._state.positive_brier_count += len(pos_indices)

    def compute(self) -> Dict[str, float]:
        s = self._state
        results = {
            'val_loss': s.total_loss / max(s.num_batches, 1),
            'mrr': s.mrr_sum / max(s.mrr_count, 1) if self.compute_mrr else 0.0,
            'positive_brier': s.positive_brier_sum / max(s.positive_brier_count, 1) if self.compute_brier else 0.0,
        }
        for k in self.k_values:
            results[f'recall@{k}'] = s.recall_hits[k] / max(s.recall_total[k], 1)
            results[f'micro_recall@{k}'] = s.micro_recall_hits[k] / max(s.micro_recall_true[k], 1)
            results[f'precision@{k}'] = s.precision_sum[k] / max(s.precision_count[k], 1)
            results[f'ndcg@{k}'] = s.ndcg_sum[k] / max(s.ndcg_count[k], 1)
        return results

print("StreamingMetrics loaded.")

StreamingMetrics loaded.


In [43]:
class GPUMemoryTracker:
    """Track GPU memory at different stages of training (from moe_flashattn_4)."""

    def __init__(self, enabled: bool = True):
        self.enabled = enabled and torch.cuda.is_available()
        self.num_gpus = torch.cuda.device_count() if self.enabled else 0
        self.records = {}

    def record(self, stage_name: str):
        if not self.enabled:
            return
        torch.cuda.synchronize()
        self.records[stage_name] = {}
        for gpu_id in range(self.num_gpus):
            allocated = torch.cuda.memory_allocated(gpu_id) / 1024**3
            reserved = torch.cuda.memory_reserved(gpu_id) / 1024**3
            peak = torch.cuda.max_memory_allocated(gpu_id) / 1024**3
            self.records[stage_name][gpu_id] = (allocated, reserved, peak)

    def reset_peak(self):
        if self.enabled:
            for gpu_id in range(self.num_gpus):
                torch.cuda.reset_peak_memory_stats(gpu_id)

    def print_gpu_use_summary(self):
        if not self.records:
            print("No GPU memory records.")
            return
        print("\n" + "="*70)
        print("GPU MEMORY SUMMARY")
        print("="*70)
        stages = list(self.records.keys())
        print(f"{'GPU':<6}", end="")
        for stage in stages:
            print(f"{stage:<20}", end="")
        print()
        print("-"*70)
        for gpu_id in range(self.num_gpus):
            print(f"GPU {gpu_id:<2}", end="")
            for stage in stages:
                alloc, _, _ = self.records[stage][gpu_id]
                print(f"{alloc:>6.2f}GB             ", end="")
            print()


def cleanup_gpu_memory(verbose=True):
    """Comprehensive GPU memory cleanup."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        if verbose:
            print("GPU Memory Status:")
            for gpu_id in range(torch.cuda.device_count()):
                total = torch.cuda.get_device_properties(gpu_id).total_mem / 1024**3
                alloc = torch.cuda.memory_allocated(gpu_id) / 1024**3
                reserved = torch.cuda.memory_reserved(gpu_id) / 1024**3
                free = total - reserved
                print(f"  GPU {gpu_id}: total={total:.2f}GB, allocated={alloc:.2f}GB, "
                      f"reserved={reserved:.2f}GB, free={free:.2f}GB")

print("GPUMemoryTracker and cleanup_gpu_memory loaded.")

GPUMemoryTracker and cleanup_gpu_memory loaded.


## Model Definition
Legacy hierarchical clinical transformer with log_softmax bug fix.

In [26]:
class LegacyTransformerModel(nn.Module):
    """
    Legacy hierarchical clinical transformer.

    Architecture (unchanged from min_transformer_train.py):
    - Daily encoder: 1 layer, 4 heads, d_ff=embedding_size, dropout=0
    - Temporal encoder: nlayers layers, nhead heads, d_ff=nhid, dropout=ndropout
    - Causal mask on temporal encoder
    - Max pooling on daily encoder output
    - Residual sum: code_sum + max_pool + gender + age + lob

    Bug fix: Returns raw logits instead of log_softmax output.
    LOB embedding included for data compatibility with moe_flashattn_4 raw tables.
    """

    def __init__(self, nhead, nhid, nlayers, dropout=0.05):
        super(LegacyTransformerModel, self).__init__()

        self.embedding_cd = nn.Embedding(cd_cnt, embedding_size)
        self.embedding_cd.weight.requires_grad = True
        self.embedding_gender_cd = nn.Embedding(4, embedding_size)
        self.embedding_gender_cd.weight.requires_grad = True
        self.embedding_age_in_months = nn.Embedding(1440, embedding_size)
        self.embedding_age_in_months.weight.requires_grad = True
        self.embedding_lob = nn.Embedding(lob_vocab, embedding_size)
        self.embedding_lob.weight.requires_grad = True

        encoder_layers_cd = TransformerEncoderLayer(embedding_size, 4, embedding_size, 0)
        self.transformer_encoder_cd = TransformerEncoder(encoder_layers_cd, 1)

        encoder_layers_dy = TransformerEncoderLayer(embedding_size, nhead, nhid, dropout)
        self.transformer_encoder_dy = TransformerEncoder(encoder_layers_dy, nlayers)

        self.mm = nn.GELU()
        self.decoder_cd = nn.Linear(embedding_size, target_cd_cnt)
        self.dropout = nn.Dropout(0.1)
        self.norm = nn.LayerNorm(embedding_size)
        self.init_weights()

    def _generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def init_weights(self):
        initrange = 0.1
        nn.init.zeros_(self.decoder_cd.weight)
        nn.init.uniform_(self.decoder_cd.weight, -initrange, initrange)

    def forward(self, x):
        gpu_batchsize = x.shape[0]
        age_in_months = x[:, :, 0]
        gender_cd = x[:, :, 1]
        lob = x[:, :, 2]

        gender_cd = self.embedding_gender_cd(gender_cd)
        age_in_months = self.embedding_age_in_months(age_in_months)
        lob_emb = self.embedding_lob(lob)

        cd = x[:, :, 3:]
        cd = self.embedding_cd(cd)
        cd_res = cd.sum(-2)
        cd = cd.reshape(gpu_batchsize * len_dy, len_cd, embedding_size)
        cd = torch.swapaxes(cd, 0, 1)
        cd = self.transformer_encoder_cd(cd)
        cd = cd.permute(1, 2, 0)
        cd = nn.MaxPool1d(len_cd)(cd)
        cd = cd.reshape(gpu_batchsize, len_dy, embedding_size)
        cd = cd_res + cd + gender_cd + age_in_months + lob_emb
        cd = self.mm(cd)
        cd = self.norm(cd)
        cd = torch.swapaxes(cd, 0, 1)

        mth_mask = self._generate_square_subsequent_mask(len_dy).to(x.device)
        cd = self.transformer_encoder_dy(cd, mth_mask)
        cd = torch.swapaxes(cd, 0, 1)
        cd = self.norm(cd)
        cd = self.dropout(cd)
        cd = self.decoder_cd(cd)
        return cd

print("LegacyTransformerModel loaded.")

LegacyTransformerModel loaded.


## Dataset & Data Loading

In [27]:
def conv_lob(ipt, length=200):
    """Convert LOB string to index list. Matches moe_flashattn_4.conv_lob."""
    lob_map = {'commercial': 1, 'medicare': 2, 'medicaid': 3}
    if not ipt or (isinstance(ipt, float) and pd.isna(ipt)):
        return [0] * length
    ipt_str = str(ipt)
    if '*' in ipt_str:
        values = ipt_str.split('*')
    else:
        values = [ipt_str] * length
    values = values[:length]
    result = []
    for val in values:
        val_clean = val.strip().lower() if val else ''
        result.append(lob_map.get(val_clean, 0))
    if result:
        last_val = result[-1] if result[-1] != 0 else 3
        while len(result) < length:
            result.append(last_val)
    else:
        result = [3] * length
    return result


class ClinicalDataset(Dataset):
    def __init__(self, df, target_col='target'):
        self.samples = []
        self.target_col = target_col
        has_lob = 'lob' in df.columns
        if minimum_mth_training > 0:
            df = df[df['dt_cnt'] >= minimum_mth_training].reset_index(drop=True)
            print(f"After filtering dt_cnt >= {minimum_mth_training}: {len(df)} samples")

        for idx in range(len(df)):
            if idx % 50000 == 0:
                print(f"  Pre-processing {idx}/{len(df)}...")
            row = df.iloc[idx]
            age = self._parse_age_gender(row['age_in_months'])
            gender = self._parse_age_gender(row['gender_cd'])
            codes = self._parse_codes(row['cd'])
            lob = conv_lob(row['lob'], len_dy) if has_lob else [0] * len_dy
            if target_col in row and pd.notna(row[target_col]):
                target_val = self._parse_target(row[target_col])
            else:
                target_val = []
            self.samples.append({
                'age': np.array(age, dtype=np.int64),
                'gender': np.array(gender, dtype=np.int64),
                'lob': np.array(lob, dtype=np.int64),
                'codes': np.array(codes, dtype=np.int64),
                'dt_cnt': int(row['dt_cnt']),
                'target': target_val,
                entity_id: row[entity_id] if entity_id in row.index else None
            })
        print(f"Pre-processing complete: {len(self.samples)} samples")

    def _parse_age_gender(self, ipt):
        ipt = ipt.split('*')
        ipt = ipt[:len_dy]
        ipt = [min(int(cd), 1439) if cd != '' else 0 for cd in ipt]
        ipt = ipt + (len_dy - len(ipt)) * [0]
        return ipt

    def _parse_codes(self, ipt):
        ipt = ipt.split('*')
        ipt = ipt[:len_dy]
        ipt = ipt + (len_dy - len(ipt)) * ['']
        ipt = [dy.split(',') for dy in ipt]
        ipt = [[int(cd) if cd != '' else 0 for cd in dy] for dy in ipt]
        ipt = [dy[:len_cd] + (len_cd - len(dy[:len_cd])) * [0] for dy in ipt]
        return ipt

    def _parse_target(self, target_str):
        days = target_str.split('*')
        days = days[:len_dy]
        result = []
        for dy in days:
            codes = []
            for cd in dy.split(','):
                try:
                    v = int(cd) if cd != '' else 0
                    if 0 <= v < target_cd_cnt:
                        codes.append(v)
                except ValueError:
                    pass
            if not codes:
                codes = [0]
            result.append(codes)
        while len(result) < len_dy:
            result.append([0])
        return result

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return {
            'age': torch.from_numpy(sample['age']),
            'gender': torch.from_numpy(sample['gender']),
            'lob': torch.from_numpy(sample['lob']),
            'codes': torch.from_numpy(sample['codes']),
            'dt_cnt': sample['dt_cnt'],
            'target': sample['target'],
            entity_id: sample[entity_id]
        }


def clinical_collate_fn(batch):
    """Custom collate that handles variable-length target lists."""
    ages = torch.stack([item['age'] for item in batch])
    genders = torch.stack([item['gender'] for item in batch])
    lobs = torch.stack([item['lob'] for item in batch])
    codes = torch.stack([item['codes'] for item in batch])
    dt_cnts = torch.tensor([item['dt_cnt'] for item in batch], dtype=torch.long)
    targets_list = [item['target'] for item in batch]
    ids_list = [item.get(entity_id) for item in batch]
    return {
        'age': ages,
        'gender': genders,
        'lob': lobs,
        'codes': codes,
        'dt_cnt': dt_cnts,
        'target': targets_list,
        entity_id: ids_list,
    }


def load_training_data(table_name: str = None, limit: int = None) -> pd.DataFrame:
    """Load training dataset from BigQuery."""
    from google.cloud import bigquery
    client = bigquery.Client(project=GCP_PROJECT)

    if table_name is None:
        table_name = BIGQUERY_TABLE_FULL

    query = f"""
    SELECT individual_id, age_in_months, gender_cd, cd, dt_cnt, lob,
           target
    FROM `{table_name}`
    """
    if limit:
        query += f" LIMIT {limit}"

    print(f"Loading data from {table_name}...")
    df = client.query(query).to_dataframe()
    print(f"Loaded {len(df):,} rows")

    member_counts = df.groupby('individual_id').size()
    single_record = member_counts[member_counts == 1].index
    df = df[df['individual_id'].isin(single_record)].copy()
    print(f"After dedup: {len(df):,} unique members")
    return df


class ClinicalDatasetLazy(Dataset):
    """Memory-efficient Dataset: stores raw strings, parses on-the-fly in __getitem__.

    For 11M samples:
      - ClinicalDataset:     ~1,440 GB RAM (pre-allocated numpy arrays + target lists)
      - ClinicalDatasetLazy: ~20-30 GB RAM (raw strings only)

    Interface contract: __getitem__ returns identical dict as ClinicalDataset,
    so clinical_collate_fn, DataLoader, and training loop require zero changes.
    """
    def __init__(self, df, target_col='target'):
        self.target_col = target_col
        has_lob = 'lob' in df.columns

        if minimum_mth_training > 0:
            df = df[df['dt_cnt'] >= minimum_mth_training].reset_index(drop=True)
            print(f"After filtering dt_cnt >= {minimum_mth_training}: {len(df)} samples")

        self.n = len(df)
        print(f"ClinicalDatasetLazy: Storing {self.n:,} samples as raw strings (lazy parsing)...")
        start = time.time()

        self.age_strs = df['age_in_months'].tolist()
        self.gender_strs = df['gender_cd'].tolist()
        self.cd_strs = df['cd'].tolist()
        self.dt_cnts = df['dt_cnt'].astype(int).tolist()
        self.ids = df[entity_id].tolist() if entity_id in df.columns else [None] * self.n

        if target_col in df.columns:
            self.target_strs = df[target_col].tolist()
        else:
            self.target_strs = [None] * self.n

        if has_lob:
            self.lob_strs = df['lob'].tolist()
        else:
            self.lob_strs = [None] * self.n

        sample_size = min(1000, self.n)
        avg_cd_len = sum(
            len(str(s)) if s and not pd.isna(s) else 0
            for s in self.cd_strs[:sample_size]
        ) / max(sample_size, 1)
        est_gb = (avg_cd_len * self.n * 1.5) / 1e9

        elapsed = time.time() - start
        print(f"  Done in {elapsed:.1f}s. Estimated string memory: ~{est_gb:.1f} GB")
        print(f"  Parsing will happen on-the-fly in __getitem__ (parallelized by DataLoader workers)")

    def _parse_age_gender(self, ipt):
        if not ipt or (isinstance(ipt, float) and pd.isna(ipt)):
            return [0] * len_dy
        ipt = str(ipt).split('*')
        ipt = ipt[:len_dy]
        ipt = [min(int(cd), 1439) if cd != '' else 0 for cd in ipt]
        ipt = ipt + (len_dy - len(ipt)) * [0]
        return ipt

    def _parse_codes(self, ipt):
        if not ipt or (isinstance(ipt, float) and pd.isna(ipt)):
            return [[0] * len_cd for _ in range(len_dy)]
        ipt = str(ipt).split('*')
        ipt = ipt[:len_dy]
        ipt = ipt + (len_dy - len(ipt)) * ['']
        ipt = [dy.split(',') for dy in ipt]
        ipt = [[int(cd) if cd != '' else 0 for cd in dy] for dy in ipt]
        ipt = [dy[:len_cd] + (len_cd - len(dy[:len_cd])) * [0] for dy in ipt]
        return ipt

    def _parse_target(self, target_str):
        if not target_str or pd.isna(target_str):
            return [[0] for _ in range(len_dy)]
        days = str(target_str).split('*')
        days = days[:len_dy]
        result = []
        for dy in days:
            codes = []
            for cd in dy.split(','):
                try:
                    v = int(cd) if cd != '' else 0
                    if 0 <= v < target_cd_cnt:
                        codes.append(v)
                except ValueError:
                    pass
            if not codes:
                codes = [0]
            result.append(codes)
        while len(result) < len_dy:
            result.append([0])
        return result

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        age = self._parse_age_gender(self.age_strs[idx])
        gender = self._parse_age_gender(self.gender_strs[idx])
        codes = self._parse_codes(self.cd_strs[idx])
        lob = conv_lob(self.lob_strs[idx], len_dy) if self.lob_strs[idx] is not None else [0] * len_dy
        target_val = self._parse_target(self.target_strs[idx])

        return {
            'age': torch.tensor(age, dtype=torch.long),
            'gender': torch.tensor(gender, dtype=torch.long),
            'lob': torch.tensor(lob, dtype=torch.long),
            'codes': torch.tensor(codes, dtype=torch.long),
            'dt_cnt': self.dt_cnts[idx],
            'target': target_val,
            entity_id: self.ids[idx],
        }


def compute_code_freq_from_strings(target_strs, sample_fraction=1.0):
    """Compute code frequencies directly from raw target strings.
    Used with ClinicalDatasetLazy to avoid materializing the full targets list.
    Matches the eager code_freq computation behavior (skips code 0).
    """
    code_freq = np.zeros(target_cd_cnt, dtype=np.int64)
    n = len(target_strs)
    if sample_fraction < 1.0:
        n_process = int(n * sample_fraction)
        indices = np.random.choice(n, n_process, replace=False)
    else:
        n_process = n
        indices = range(n)

    print(f"  Computing code frequencies from {n_process:,} target strings...")
    for count, idx in enumerate(indices):
        target_str = target_strs[idx]
        if not target_str or pd.isna(target_str):
            continue
        for day_str in str(target_str).split('*')[:len_dy]:
            if not day_str:
                continue
            for code_str in day_str.split(','):
                try:
                    v = int(code_str) if code_str else 0
                    if 0 < v < target_cd_cnt:
                        code_freq[v] += 1
                except ValueError:
                    pass
        if (count + 1) % 1_000_000 == 0:
            print(f"    {count + 1:,}/{n_process:,} processed...")

    non_zero = np.sum(code_freq > 0)
    print(f"  Non-zero target codes: {non_zero:,} / {target_cd_cnt}")
    return code_freq


print("Dataset and data loading functions loaded.")

Dataset and data loading functions loaded.


## Batch Metrics & Loss Computation
Adapted from `moe_flashattn_4.compute_batch_metrics_lightweight` for the legacy model (no BaseConfig dependency).

In [28]:
def compute_batch_metrics_legacy(
    output: torch.Tensor,
    targets_flat: List[List[int]],
    dt_cnt: List[int],
) -> Dict[str, float]:
    """
    Lightweight batch metrics matching moe_flashattn_4 format.
    Adapted for legacy model (no config object, direct parameters).

    Metrics: recall@K, precision@K, micro_recall@K, NDCG@20, positive_brier
    """
    with torch.no_grad():
        bs = len(dt_cnt)
        output_flat = output.reshape(bs * len_dy, target_cd_cnt)

        valid_outputs = []
        valid_y = []
        for j in range(bs):
            valid_days = min(int(dt_cnt[j]), len_dy)
            if valid_days <= 0:
                continue
            start_idx = len_dy * j
            end_idx = start_idx + valid_days
            valid_outputs.append(output_flat[start_idx:end_idx])
            y_start = len_dy * j
            y_end = y_start + valid_days
            valid_y.extend(targets_flat[y_start:y_end])

        if len(valid_outputs) == 0:
            return {
                'recall@1': 0.0, 'recall@5': 0.0, 'recall@10': 0.0, 'recall@20': 0.0, 'recall@50': 0.0,
                'precision@5': 0.0, 'precision@10': 0.0, 'precision@20': 0.0, 'precision@50': 0.0,
                'micro_recall@1': 0.0, 'micro_recall@10': 0.0, 'micro_recall@20': 0.0,
                'ndcg@20': 0.0, 'positive_brier': 0.0
            }

        predictions = torch.cat(valid_outputs)
        num_samples = len(predictions)
        metrics = {}
        sorted_indices = torch.argsort(predictions, dim=-1, descending=True)

        for k in [1, 5, 10, 20, 50]:
            top_k_preds = sorted_indices[:, :k]
            correct, total = 0, 0
            for i, target_codes in enumerate(valid_y):
                true_codes = [c for c in target_codes if c != 0]
                if len(true_codes) > 0:
                    total += 1
                    if any(code in top_k_preds[i].tolist() for code in true_codes):
                        correct += 1
            metrics[f'recall@{k}'] = correct / total if total > 0 else 0.0

        for k in [5, 10, 20, 50]:
            top_k_preds = sorted_indices[:, :k]
            precisions = []
            for i, target_codes in enumerate(valid_y):
                true_codes = set(c for c in target_codes if c != 0)
                if len(true_codes) > 0:
                    hits = sum(1 for code in top_k_preds[i].tolist() if code in true_codes)
                    precisions.append(hits / k)
            metrics[f'precision@{k}'] = np.mean(precisions) if precisions else 0.0

        for k in [1, 5, 10, 20]:
            top_k_preds = sorted_indices[:, :k]
            total_hits, total_true = 0, 0
            for i, target_codes in enumerate(valid_y):
                true_codes = set(c for c in target_codes if c != 0)
                if len(true_codes) > 0:
                    total_true += len(true_codes)
                    pred_set = set(top_k_preds[i].tolist())
                    total_hits += len(true_codes & pred_set)
            metrics[f'micro_recall@{k}'] = total_hits / total_true if total_true > 0 else 0.0

        k = 20
        disc = 1.0 / np.log2(np.arange(2, k + 2))
        ndcg_scores = []
        for i, target_codes in enumerate(valid_y):
            true_codes = set(c for c in target_codes if c != 0)
            if len(true_codes) == 0:
                continue
            top_k_preds = sorted_indices[i, :k].tolist()
            dcg = sum(disc[rank] for rank, pred in enumerate(top_k_preds) if pred in true_codes)
            num_relevant = min(len(true_codes), k)
            idcg = sum(disc[:num_relevant])
            ndcg_scores.append(dcg / idcg if idcg > 0 else 0.0)
        metrics['ndcg@20'] = np.mean(ndcg_scores) if ndcg_scores else 0.0

        probs = torch.sigmoid(predictions)
        positive_probs = []
        for i, target_codes in enumerate(valid_y):
            for code in target_codes:
                if 0 < code < target_cd_cnt:
                    positive_probs.append(probs[i, code].item())
        if len(positive_probs) > 0:
            positive_probs = np.array(positive_probs)
            metrics['positive_brier'] = float(np.mean((positive_probs - 1.0) ** 2))
        else:
            metrics['positive_brier'] = 0.0

        return metrics


def compute_loss_legacy(output, targets_flat, dt_cnt, criterion):
    """Compute BCE loss over valid timesteps (matching legacy pattern)."""
    bs = output.shape[0]
    output_flat = output.reshape(-1, target_cd_cnt)

    valid_outputs = torch.cat(
        [output_flat[len_dy * i:len_dy * i + dt_cnt[i], :] for i in range(bs)],
        dim=0
    )

    valid_targets = []
    for i in range(bs):
        start = len_dy * i
        end = start + dt_cnt[i]
        valid_targets.extend(targets_flat[start:end])

    y_cd = torch.zeros(len(valid_outputs), target_cd_cnt, device=output.device)
    for j in range(len(valid_outputs)):
        for k in valid_targets[j]:
            if k != 0:
                y_cd[j, k] = 1

    return criterion(valid_outputs, y_cd)

print("Batch metrics and loss computation loaded.")

Batch metrics and loss computation loaded.


## Training & Evaluation
Full training loop with batch-level metrics, gradient tier analysis, and loss tracking.
Validation via StreamingMetrics for comparability with moe_flashattn_4.

In [29]:
def currentTime():
    tz = pytz.timezone("America/New_York")
    return datetime.now(tz).strftime("%D %H:%M:%S")

def train_epoch(
    model, dataloader, optimizer, criterion,
    epoch: int = 0,
    log_interval: int = 100,
    global_step: int = 0,
    loss_tracker: Optional[LossTracker] = None,
    metrics_logger: Optional[MetricsLogger] = None,
    logger: Optional[logging.Logger] = None,
    gradient_tier_analyzer: Optional[GradientTierAnalyzer] = None,
    accumulation_steps: int = 1,
    track_gpu_memory: bool = True,
    scaler: Optional[torch.cuda.amp.GradScaler] = None,
    on_optimizer_step: Optional[Callable] = None,
) -> Dict[str, Any]:
    """
    Train one epoch with comprehensive metrics tracking.
    Matches moe_flashattn_4 train_epoch output format.

    Returns dict with: train_loss, aux_loss (always 0 for legacy), loss tracker summary,
    averaged batch metrics, gradient tier summary, global_step.
    """
    model.train()
    gpu_tracker = GPUMemoryTracker(enabled=track_gpu_memory)
    num_batches = len(dataloader)
    total_loss = 0.0
    batch_metrics_buffer = []
    gradient_tier_buffer = []
    use_amp = scaler is not None

    epoch_start_time = time.time()
    data_load_time = 0.0
    forward_time = 0.0
    backward_time = 0.0
    optimizer_time = 0.0
    samples_processed = 0

    if loss_tracker is None:
        loss_tracker = LossTracker()

    accumulated_loss = 0.0
    accumulation_counter = 0
    data_start = time.time()

    for batch_idx, batch in enumerate(dataloader):
        data_load_time += time.time() - data_start

        should_track = track_gpu_memory and batch_idx in [2, 50, 100]
        if should_track:
            gpu_tracker.reset_peak()
            print(f"\n  GPU tracking for batch {batch_idx}")

        if batch_idx % log_interval == 0:
            print(f'  Batch {batch_idx}/{num_batches}  {currentTime()}')

        if batch_idx == 0 and torch.cuda.is_available():
            num_gpus = torch.cuda.device_count()
            if num_gpus > 1:
                print(f"\n  GPU UTILIZATION CHECK (Batch 0):")
                for gpu_id in range(num_gpus):
                    mem_alloc = torch.cuda.memory_allocated(gpu_id) / 1024**3
                    mem_reserved = torch.cuda.memory_reserved(gpu_id) / 1024**3
                    print(f"   GPU {gpu_id}: {mem_alloc:.2f} GB allocated, {mem_reserved:.2f} GB reserved")

        if accumulation_counter == 0:
            optimizer.zero_grad(set_to_none=True)

        age = batch['age'].to(device, non_blocking=True)
        gender = batch['gender'].to(device, non_blocking=True)
        lob = batch['lob'].to(device, non_blocking=True)
        codes = batch['codes'].to(device, non_blocking=True)
        dt_cnt = batch['dt_cnt']
        targets = batch['target']

        samples_processed += age.shape[0]

        x = torch.cat([age.unsqueeze(-1), gender.unsqueeze(-1), lob.unsqueeze(-1), codes], dim=-1)

        if should_track:
            gpu_tracker.record("1_after_data_to_gpu")

        fwd_start = time.time()
        with torch.cuda.amp.autocast(enabled=use_amp):
            output = model(x)
        forward_time += time.time() - fwd_start

        if should_track:
            gpu_tracker.record("2_after_forward")

        targets_flat = [item for sublist in targets for item in sublist]
        dt_cnt_list = dt_cnt.tolist() if isinstance(dt_cnt, torch.Tensor) else dt_cnt

        with torch.cuda.amp.autocast(enabled=use_amp):
            loss = compute_loss_legacy(output, targets_flat, dt_cnt_list, criterion)
        loss_scalar = loss.item()
        total_loss += loss_scalar

        scaled_loss = loss / accumulation_steps

        bwd_start = time.time()
        if use_amp:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()
        backward_time += time.time() - bwd_start

        if should_track:
            gpu_tracker.record("3_after_backward")

        if gradient_tier_analyzer is not None and batch_idx % log_interval == 0:
            tier_metrics = gradient_tier_analyzer.log_batch(model, batch_idx)
            if tier_metrics:
                gradient_tier_buffer.append(tier_metrics)
                if len(gradient_tier_buffer) > 100:
                    gradient_tier_buffer = gradient_tier_buffer[-100:]

        accumulated_loss += loss_scalar
        accumulation_counter += 1

        if accumulation_counter >= accumulation_steps:
            opt_start = time.time()
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
                optimizer.step()
            optimizer_time += time.time() - opt_start

            loss_tracker.log_batch(accumulated_loss / accumulation_steps, global_step)
            global_step += 1
            # --- Sub-epoch callback (e.g., for early stopping validation) ---
            if on_optimizer_step is not None:
                stop_signal = on_optimizer_step(
                    global_step=global_step,
                    model=model,
                    loss_tracker=loss_tracker,
                    epoch=epoch,
                    batch_idx=batch_idx,
                )
                if stop_signal:
                    epoch_time = time.time() - epoch_start_time
                    avg_loss = total_loss / max(batch_idx + 1, 1)
                    epoch_metrics = {
                        'train_loss': avg_loss,
                        'aux_loss': 0.0,
                        'global_step': global_step,
                        'num_batches': batch_idx + 1,
                        'epoch_time_s': epoch_time,
                        'data_load_time_s': data_load_time,
                        'forward_time_s': forward_time,
                        'backward_time_s': backward_time,
                        'optimizer_time_s': optimizer_time,
                        'samples_processed': samples_processed,
                        'throughput_samples_per_sec': samples_processed / epoch_time if epoch_time > 0 else 0,
                        'throughput_batches_per_sec': (batch_idx + 1) / epoch_time if epoch_time > 0 else 0,
                        'early_stopped': True,
                        'stopped_at_batch': batch_idx,
                    }
                    if torch.cuda.is_available():
                        epoch_metrics['gpu_memory_peak_gib'] = torch.cuda.max_memory_allocated() / 1024**3
                        epoch_metrics['gpu_memory_allocated_gib'] = torch.cuda.memory_allocated() / 1024**3
                    loss_summary = loss_tracker.get_epoch_summary()
                    epoch_metrics.update(loss_summary)
                    if batch_metrics_buffer:
                        for key in batch_metrics_buffer[0].keys():
                            epoch_metrics[f'train_{key}'] = np.mean([m[key] for m in batch_metrics_buffer])
                    if gradient_tier_analyzer is not None:
                        tier_epoch = gradient_tier_analyzer.aggregate_epoch()
                        epoch_metrics.update(tier_epoch)
                    print(f'  EARLY STOP at batch {batch_idx}/{num_batches}. '
                          f'Avg loss: {avg_loss:.4f} | Time: {epoch_time:.1f}s')
                    return epoch_metrics
            accumulated_loss = 0.0
            accumulation_counter = 0

        if should_track:
            gpu_tracker.print_gpu_use_summary()

        if batch_idx % log_interval == 0:
            with torch.no_grad():
                batch_metrics = compute_batch_metrics_legacy(
                    output.detach().float(), targets_flat, dt_cnt_list
                )
                batch_metrics_buffer.append(batch_metrics)
                if len(batch_metrics_buffer) > 100:
                    batch_metrics_buffer = batch_metrics_buffer[-100:]

                batch_log_msg = (
                    f"    Loss: {loss_scalar:.4f} | "
                    f"R@10: {batch_metrics['recall@10']:.3f} | "
                    f"R@20: {batch_metrics['recall@20']:.3f} | "
                    f"uR@10: {batch_metrics['micro_recall@10']:.3f} | "
                    f"P@10: {batch_metrics['precision@10']:.3f} | "
                    f"NDCG@20: {batch_metrics['ndcg@20']:.3f} | "
                    f"PosBrier: {batch_metrics['positive_brier']:.4f}"
                )
                print(batch_log_msg)
                if logger:
                    logger.debug(batch_log_msg)

                batch_entry = {
                    'global_step': global_step,
                    'loss': loss_scalar,
                    **batch_metrics
                }

                if gradient_tier_buffer:
                    latest_tier = gradient_tier_buffer[-1]
                    batch_entry.update({
                        f'grad_{k}': v for k, v in latest_tier.items()
                    })

                if metrics_logger:
                    metrics_logger.log_batch(epoch=epoch, batch=batch_idx, metrics=batch_entry)

        del output, x, age, gender, lob, codes, loss, scaled_loss

        if batch_idx % 500 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        data_start = time.time()

    if accumulation_counter > 0:
        if use_amp:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
            optimizer.step()
        loss_tracker.log_batch(accumulated_loss / accumulation_counter, global_step)
        global_step += 1
        if on_optimizer_step is not None:
            on_optimizer_step(
                global_step=global_step,
                model=model,
                loss_tracker=loss_tracker,
                epoch=epoch,
                batch_idx=num_batches - 1,
            )

    epoch_time = time.time() - epoch_start_time
    avg_loss = total_loss / max(num_batches, 1)

    epoch_metrics = {
        'train_loss': avg_loss,
        'aux_loss': 0.0,
        'global_step': global_step,
        'num_batches': num_batches,
        'epoch_time_s': epoch_time,
        'data_load_time_s': data_load_time,
        'forward_time_s': forward_time,
        'backward_time_s': backward_time,
        'optimizer_time_s': optimizer_time,
        'samples_processed': samples_processed,
        'throughput_samples_per_sec': samples_processed / epoch_time if epoch_time > 0 else 0,
        'throughput_batches_per_sec': num_batches / epoch_time if epoch_time > 0 else 0,
    }

    if torch.cuda.is_available():
        epoch_metrics['gpu_memory_peak_gib'] = torch.cuda.max_memory_allocated() / 1024**3
        epoch_metrics['gpu_memory_allocated_gib'] = torch.cuda.memory_allocated() / 1024**3

    loss_summary = loss_tracker.get_epoch_summary()
    epoch_metrics.update(loss_summary)

    if batch_metrics_buffer:
        for key in batch_metrics_buffer[0].keys():
            epoch_metrics[f'train_{key}'] = np.mean([m[key] for m in batch_metrics_buffer])

    if gradient_tier_analyzer is not None:
        tier_epoch = gradient_tier_analyzer.aggregate_epoch()
        epoch_metrics.update(tier_epoch)

    epoch_metrics['global_step'] = global_step

    compute_time = forward_time + backward_time + optimizer_time
    total_accounted = data_load_time + compute_time
    if total_accounted > 0:
        epoch_metrics['data_load_pct'] = (data_load_time / total_accounted) * 100
        epoch_metrics['forward_pct'] = (forward_time / total_accounted) * 100
        epoch_metrics['backward_pct'] = (backward_time / total_accounted) * 100
        epoch_metrics['optimizer_pct'] = (optimizer_time / total_accounted) * 100

    print(f'  Training complete. Avg loss: {avg_loss:.4f} | '
          f'Time: {epoch_time:.1f}s | '
          f'Throughput: {epoch_metrics["throughput_samples_per_sec"]:.1f} samples/sec | '
          f'Data load: {epoch_metrics.get("data_load_pct", 0):.1f}%')
    return epoch_metrics

In [30]:
def evaluate(
    model, dataloader, criterion,
    max_batches: Optional[int] = None,
    verbose: bool = False,
    k_values: Tuple[int, ...] = (1, 5, 10, 20),
    use_amp: bool = False,
) -> Dict[str, float]:
    """
    Memory-efficient evaluation using StreamingMetrics.
    Matches moe_flashattn_4 evaluate() output format.
    """
    model.eval()
    num_batches = len(dataloader)
    batches_to_process = min(num_batches, max_batches) if max_batches else num_batches

    if batches_to_process == 0:
        metrics = {'val_loss': 0.0, 'mrr': 0.0, 'positive_brier': 0.0}
        for k in k_values:
            metrics[f'recall@{k}'] = 0.0
            metrics[f'micro_recall@{k}'] = 0.0
            metrics[f'precision@{k}'] = 0.0
            metrics[f'ndcg@{k}'] = 0.0
        return metrics

    metrics_tracker = StreamingMetrics(
        k_values=k_values, compute_mrr=True, compute_brier=True,
        vocab_size=target_cd_cnt
    )

    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx >= batches_to_process:
                break
            if verbose and batch_idx % 100 == 0:
                print(f"    Val batch {batch_idx}/{batches_to_process}")

            age = batch['age'].to(device, non_blocking=True)
            gender = batch['gender'].to(device, non_blocking=True)
            lob = batch['lob'].to(device, non_blocking=True)
            codes = batch['codes'].to(device, non_blocking=True)
            dt_cnt = batch['dt_cnt']
            targets = batch['target']

            x = torch.cat([age.unsqueeze(-1), gender.unsqueeze(-1), lob.unsqueeze(-1), codes], dim=-1)

            with torch.cuda.amp.autocast(enabled=use_amp):
                output = model(x)

            output_f32 = output.float()

            targets_flat = [item for sublist in targets for item in sublist]
            dt_cnt_list = dt_cnt.tolist() if isinstance(dt_cnt, torch.Tensor) else dt_cnt
            loss = compute_loss_legacy(output_f32, targets_flat, dt_cnt_list, criterion)
            metrics_tracker.update_loss(loss.item())

            bs = output_f32.shape[0]
            output_flat = output_f32.view(bs * len_dy, target_cd_cnt)

            valid_outputs = []
            valid_targets = []
            for j in range(bs):
                valid_days = min(int(dt_cnt_list[j]), len_dy)
                if valid_days <= 0:
                    continue
                out_start = len_dy * j
                out_end = out_start + valid_days
                valid_outputs.append(output_flat[out_start:out_end])
                y_start = len_dy * j
                y_end = y_start + valid_days
                valid_targets.extend(targets_flat[y_start:y_end])

            if valid_outputs:
                preds = torch.cat(valid_outputs)
                metrics_tracker.update(preds, valid_targets)
                del preds, valid_outputs, valid_targets

            del output, output_f32
            if batch_idx % 1000 == 0:
                gc.collect()

    results = metrics_tracker.compute()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

    return results
print("evaluate loaded.")

evaluate loaded.


In [31]:
def compute_training_time_metrics(
    total_train_time: float,
    num_epochs: int,
    num_samples: int,
    num_tokens: int,
    batch_size: int,
    data_load_time: float = 0.0,
    forward_time: float = 0.0,
    backward_time: float = 0.0
) -> Dict[str, float]:
    """Training time and throughput metrics (from moe_flashattn_4)."""
    metrics = {}
    metrics['total_train_time_sec'] = total_train_time
    metrics['time_per_epoch_sec'] = total_train_time / num_epochs if num_epochs > 0 else 0
    metrics['time_per_sample_ms'] = (total_train_time / num_samples) * 1000 if num_samples > 0 else 0
    metrics['samples_per_sec'] = num_samples / total_train_time if total_train_time > 0 else 0
    metrics['tokens_per_sec'] = num_tokens / total_train_time if total_train_time > 0 else 0
    metrics['batches_per_sec'] = (num_samples / batch_size) / total_train_time if total_train_time > 0 and batch_size > 0 else 0
    if data_load_time > 0 or forward_time > 0:
        total_profiled = data_load_time + forward_time + backward_time
        if total_profiled > 0:
            metrics['data_load_percent'] = (data_load_time / total_profiled) * 100
            metrics['forward_percent'] = (forward_time / total_profiled) * 100
            metrics['backward_percent'] = (backward_time / total_profiled) * 100
    metrics['steps_per_sec'] = (num_samples / batch_size) / total_train_time if total_train_time > 0 and batch_size > 0 else 0
    return metrics


def compute_cost_metrics(
    training_time_sec: float,
    num_epochs: int,
    gpu_type: str = "T4",
    num_gpus: int = 4,
) -> Dict[str, float]:
    """Training cost estimation (from moe_flashattn_4)."""
    metrics = {}
    gpu_hourly_rates = {'T4': 0.35, 'V100': 2.48, 'A100': 3.67, 'L4': 0.70}
    rate_per_gpu = gpu_hourly_rates.get(gpu_type, 0.35)
    rate_total = rate_per_gpu * num_gpus
    training_hours = training_time_sec / 3600
    metrics['cost_usd'] = training_hours * rate_total
    metrics['cost_per_epoch_usd'] = metrics['cost_usd'] / max(num_epochs, 1)
    for n_proj in [10, 50, 100, 200]:
        cost_proj = (training_time_sec / max(num_epochs, 1)) * n_proj / 3600 * rate_total
        metrics[f'projected_cost_{n_proj}epochs_usd'] = cost_proj
    return metrics

print("Training resource metrics functions loaded.")

Training resource metrics functions loaded.


## Checkpoint Management & Embedding Extraction

In [32]:
def save_checkpoint_local(model, optimizer, scheduler, epoch, val_loss, filepath):
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    checkpoint = {
        'timestamp': str(currentTime()),
        'model': model.module.state_dict() if parallel and isinstance(model, nn.DataParallel) else model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler else None,
        'epoch': epoch,
        'val_loss': val_loss,
        'config': {
            'batch_size': batch_size, 'embedding_size': embedding_size,
            'nhid': nhid, 'nhead': nhead, 'nlayers': nlayers,
            'ndropout': ndropout, 'cd_cnt': cd_cnt, 'target_cd_cnt': target_cd_cnt,
            'lob_vocab': lob_vocab, 'learning_rate': LEARNING_RATE, 'gradient_clip': GRADIENT_CLIP,
        }
    }
    torch.save(checkpoint, filepath)
    print(f"  Checkpoint saved to {filepath}")


def save_checkpoint_gcs(model, optimizer, scheduler, epoch, val_loss, filename='checkpoint_latest.pt'):
    from google.cloud import storage
    import joblib
    from io import BytesIO
    checkpoint = {
        'timestamp': str(currentTime()),
        'model': model.module.state_dict() if parallel and isinstance(model, nn.DataParallel) else model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler else None,
        'epoch': epoch,
        'val_loss': val_loss
    }
    blob = storage.Client().bucket(GCS_BUCKET).blob(os.path.join(MODEL_PATH, filename))
    buf = BytesIO()
    joblib.dump(checkpoint, buf)
    buf.seek(0)
    blob.upload_from_file(buf)
    print(f"  GCS checkpoint saved: gs://{GCS_BUCKET}/{MODEL_PATH}/{filename}")


def load_checkpoint(filepath, model, optimizer=None, scheduler=None):
    checkpoint = torch.load(filepath, map_location=device, weights_only=False)
    if parallel and isinstance(model, nn.DataParallel):
        model.module.load_state_dict(checkpoint['model'])
    else:
        model.load_state_dict(checkpoint['model'])
    if optimizer is not None and 'optimizer' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer'])
    if scheduler is not None and 'scheduler' in checkpoint and checkpoint['scheduler']:
        scheduler.load_state_dict(checkpoint['scheduler'])
    return checkpoint.get('epoch', 0), checkpoint.get('val_loss', float('inf'))


def extract_embeddings(model, data_df, extraction_batch_size=512):
    """
    Extract member-level embeddings from the last valid timestep.
    Uses forward hook on transformer_encoder_dy.
    """
    model.eval()
    activation = {}

    def get_activation(name):
        def hook(model_hook, input_hook, output_hook):
            activation[name] = output_hook.detach()
        return hook

    actual_model = model.module if isinstance(model, nn.DataParallel) else model
    handle = actual_model.transformer_encoder_dy.register_forward_hook(
        get_activation('transformer_encoder_dy')
    )

    dsize = data_df.shape[0]
    nbatch = int(dsize / extraction_batch_size)
    if dsize - nbatch * extraction_batch_size > 0:
        k = extraction_batch_size - (dsize - nbatch * extraction_batch_size)
        data_df = pd.concat([data_df, pd.concat([data_df.head(1)] * k, ignore_index=True)])
    data_df = data_df.reset_index(drop=True)
    nbatch = int(data_df.shape[0] / extraction_batch_size)

    def _conv_cd(ipt):
        ipt = ipt.split('*')[:len_dy]
        ipt = ipt + (len_dy - len(ipt)) * ['']
        ipt = [dy.split(',') for dy in ipt]
        ipt = [[int(cd) if cd != '' else 0 for cd in dy] for dy in ipt]
        ipt = [dy[:len_cd] + (len_cd - len(dy[:len_cd])) * [0] for dy in ipt]
        return ipt

    def _conv_age_gender(ipt):
        ipt = ipt.split('*')[:len_dy]
        ipt = [min(int(cd), 1439) for cd in ipt]
        ipt = ipt + (len_dy - len(ipt)) * [0]
        return ipt

    ys = []
    with torch.no_grad():
        for i in tqdm(range(nbatch), desc="Extracting embeddings"):
            batch_df = data_df.iloc[i * extraction_batch_size:i * extraction_batch_size + extraction_batch_size, :]
            ages = torch.tensor([_conv_age_gender(v) for v in batch_df['age_in_months'].tolist()]).to(device)
            genders = torch.tensor([_conv_age_gender(v) for v in batch_df['gender_cd'].tolist()]).to(device)
            lobs_t = torch.tensor([conv_lob(v, len_dy) for v in (batch_df['lob'].tolist() if 'lob' in batch_df.columns else [''] * len(batch_df))]).to(device)
            codes_t = torch.tensor([_conv_cd(v) for v in batch_df['cd'].tolist()]).to(device)
            dt_cnts = batch_df['dt_cnt'].tolist()

            x = torch.cat([ages.unsqueeze(-1), genders.unsqueeze(-1), lobs_t.unsqueeze(-1), codes_t], dim=-1)
            _ = model(x)

            enc_out = activation['transformer_encoder_dy']
            embeddings = torch.stack([
                enc_out[dt_cnts[j], j, :] for j in range(extraction_batch_size)
            ])
            ys.append(embeddings)

    handle.remove()
    ys = torch.cat(ys).cpu().numpy()
    result = pd.DataFrame(ys, columns=[f'emb{i}' for i in range(embedding_size)])
    result[entity_id] = data_df[entity_id].values
    result = result.head(dsize)
    return result

print("Checkpoint management and embedding extraction loaded.")

Checkpoint management and embedding extraction loaded.


## Test Cases
Validates every functional component before running full training.

In [11]:
def _make_synthetic_data(n_samples=20, seed=42):
    """Generate synthetic data matching the BigQuery schema for testing."""
    np.random.seed(seed)
    rows = []
    for i in range(n_samples):
        n_days = np.random.randint(180, 200)
        ages = '*'.join([str(np.random.randint(200, 800)) for _ in range(n_days)])
        genders = '*'.join([str(np.random.randint(1, 3)) for _ in range(n_days)])
        cds = '*'.join(
            [','.join([str(np.random.randint(1, min(cd_cnt, 1000))) for _ in range(np.random.randint(5, 20))])
             for _ in range(n_days)]
        )
        tgts = '*'.join(
            [','.join([str(np.random.randint(1, min(target_cd_cnt, 500))) for _ in range(np.random.randint(1, 5))])
             for _ in range(n_days)]
        )
        lob_choices = ['Commercial', 'Medicare', 'Medicaid']
        lob_val = np.random.choice(lob_choices)
        lobs = '*'.join([lob_val] * n_days)
        rows.append({
            'individual_id': f'test_member_{i}',
            'age_in_months': ages,
            'gender_cd': genders,
            'cd': cds,
            'dt_cnt': n_days,
            'lob': lobs,
            'target': tgts
        })
    return pd.DataFrame(rows)


df_synthetic = _make_synthetic_data(20)
print(f"Synthetic data: {len(df_synthetic)} rows, columns: {list(df_synthetic.columns)}")

Synthetic data: 20 rows, columns: ['individual_id', 'age_in_months', 'gender_cd', 'cd', 'dt_cnt', 'lob', 'target']


In [12]:
# TEST 1: Architecture Parity
print("TEST 1: Architecture Parity")
print("-" * 60)
model_test = LegacyTransformerModel(nhead, nhid, nlayers, ndropout)
params = {name: p.shape for name, p in model_test.named_parameters()}

print("Layer-by-layer parameter shapes:")
total = 0
for name, shape in params.items():
    n = 1
    for s in shape:
        n *= s
    total += n
    print(f"  {name:<50} {str(shape):<25} {n:>12,}")
print(f"\n  {'TOTAL':<50} {'':25} {total:>12,}")

assert model_test.transformer_encoder_cd.layers[0].self_attn.num_heads == 4, "Daily encoder should have 4 heads"
assert len(model_test.transformer_encoder_cd.layers) == 1, "Daily encoder should have 1 layer"
assert len(model_test.transformer_encoder_dy.layers) == 6, "Temporal encoder should have 6 layers"
assert model_test.transformer_encoder_dy.layers[0].self_attn.num_heads == 16, "Temporal encoder should have 16 heads"
assert model_test.decoder_cd.out_features == target_cd_cnt, f"Output should be {target_cd_cnt}"

age_t = torch.randint(0, 1440, (2, len_dy))
gender_t = torch.randint(0, 4, (2, len_dy))
lob_t = torch.randint(0, lob_vocab, (2, len_dy))
codes_t = torch.randint(0, min(cd_cnt, 1000), (2, len_dy, len_cd))
x_test = torch.cat([age_t.unsqueeze(-1), gender_t.unsqueeze(-1), lob_t.unsqueeze(-1), codes_t], dim=-1)
assert x_test.shape == (2, len_dy, 3 + len_cd), f"Input shape: {x_test.shape}"
out = model_test(x_test)
assert out.shape == (2, len_dy, target_cd_cnt), f"Output shape mismatch: {out.shape}"
assert (out > 0).any(), "Output should contain positive values (raw logits, not log_softmax)"
print("\nAll architecture checks passed!")
del model_test, x_test, out

TEST 1: Architecture Parity
------------------------------------------------------------
Layer-by-layer parameter shapes:
  embedding_cd.weight                                torch.Size([75516, 256])    19,332,096
  embedding_gender_cd.weight                         torch.Size([4, 256])             1,024
  embedding_age_in_months.weight                     torch.Size([1440, 256])        368,640
  embedding_lob.weight                               torch.Size([4, 256])             1,024
  transformer_encoder_cd.layers.0.self_attn.in_proj_weight torch.Size([768, 256])         196,608
  transformer_encoder_cd.layers.0.self_attn.in_proj_bias torch.Size([768])                  768
  transformer_encoder_cd.layers.0.self_attn.out_proj.weight torch.Size([256, 256])          65,536
  transformer_encoder_cd.layers.0.self_attn.out_proj.bias torch.Size([256])                  256
  transformer_encoder_cd.layers.0.linear1.weight     torch.Size([256, 256])          65,536
  transformer_encoder_cd.lay

In [13]:
# TEST 2: Dataset Processing
print("\nTEST 2: Dataset Processing")
print("-" * 60)
dataset_test = ClinicalDataset(df_synthetic, target_col='target')
assert len(dataset_test) > 0, "Dataset should not be empty"

sample = dataset_test[0]
assert sample['age'].shape == (len_dy,), f"Age shape: {sample['age'].shape}"
assert sample['gender'].shape == (len_dy,), f"Gender shape: {sample['gender'].shape}"
assert sample['lob'].shape == (len_dy,), f"LOB shape: {sample['lob'].shape}"
assert sample['codes'].shape == (len_dy, len_cd), f"Codes shape: {sample['codes'].shape}"
assert isinstance(sample['dt_cnt'], int), "dt_cnt should be int"
assert isinstance(sample['target'], list), "target should be list"
assert sample['age'].max() <= 1439, "Age should be clipped to 1439"
assert sample['lob'].max() <= 3, "LOB should be 0-3"
assert sample['lob'].min() >= 0, "LOB should be >= 0"
print(f"  Sample shapes: age={sample['age'].shape}, gender={sample['gender'].shape}, lob={sample['lob'].shape}, codes={sample['codes'].shape}")
print(f"  dt_cnt={sample['dt_cnt']}, target days={len(sample['target'])}")
print("Dataset processing passed!")

# TEST 3: DataLoader
print("\nTEST 3: DataLoader")
print("-" * 60)
loader_test = DataLoader(dataset_test, batch_size=4, shuffle=True, drop_last=True, collate_fn=clinical_collate_fn)
batch_test = next(iter(loader_test))
assert batch_test['age'].shape == (4, len_dy), f"Batch age shape: {batch_test['age'].shape}"
assert batch_test['lob'].shape == (4, len_dy), f"Batch lob shape: {batch_test['lob'].shape}"
assert batch_test['codes'].shape == (4, len_dy, len_cd), f"Batch codes shape: {batch_test['codes'].shape}"
print(f"  Batch shapes OK: age={batch_test['age'].shape}, lob={batch_test['lob'].shape}, codes={batch_test['codes'].shape}")
print("DataLoader passed!")
del loader_test, batch_test


TEST 2: Dataset Processing
------------------------------------------------------------
After filtering dt_cnt >= 180: 20 samples
  Pre-processing 0/20...
Pre-processing complete: 20 samples
  Sample shapes: age=torch.Size([200]), gender=torch.Size([200]), lob=torch.Size([200]), codes=torch.Size([200, 80])
  dt_cnt=186, target days=200
Dataset processing passed!

TEST 3: DataLoader
------------------------------------------------------------
  Batch shapes OK: age=torch.Size([4, 200]), lob=torch.Size([4, 200]), codes=torch.Size([4, 200, 80])
DataLoader passed!


In [14]:
# TEST 4: Forward Pass & Loss
print("\nTEST 4: Forward Pass & Loss")
print("-" * 60)
model_test = LegacyTransformerModel(nhead, nhid, nlayers, ndropout).to(device)
loader_test = DataLoader(dataset_test, batch_size=4, shuffle=False, drop_last=True, collate_fn=clinical_collate_fn)
batch_test = next(iter(loader_test))

age_t = batch_test['age'].to(device)
gender_t = batch_test['gender'].to(device)
lob_t = batch_test['lob'].to(device)
codes_t = batch_test['codes'].to(device)
x_t = torch.cat([age_t.unsqueeze(-1), gender_t.unsqueeze(-1), lob_t.unsqueeze(-1), codes_t], dim=-1)

with torch.no_grad():
    output_t = model_test(x_t)
assert output_t.shape == (4, len_dy, target_cd_cnt), f"Output shape: {output_t.shape}"
assert torch.isfinite(output_t).all(), "Output should be finite"
print(f"  Forward pass OK: output shape={output_t.shape}")

criterion_test = nn.BCEWithLogitsLoss()
targets_flat_t = [item for sublist in batch_test['target'] for item in sublist]
dt_cnt_t = batch_test['dt_cnt'].tolist() if isinstance(batch_test['dt_cnt'], torch.Tensor) else batch_test['dt_cnt']
loss_t = compute_loss_legacy(output_t, targets_flat_t, dt_cnt_t, criterion_test)
assert loss_t.ndim == 0 and torch.isfinite(loss_t), f"Loss should be scalar and finite: {loss_t}"
print(f"  Loss computation OK: loss={loss_t.item():.4f}")
print("Forward pass & loss passed!")
del model_test, loader_test, batch_test, output_t, loss_t


TEST 4: Forward Pass & Loss
------------------------------------------------------------
  Forward pass OK: output shape=torch.Size([4, 200, 6297])
  Loss computation OK: loss=0.7932
Forward pass & loss passed!


In [15]:
# TEST 5: Training Step (1 batch)
print("\nTEST 5: Training Step")
print("-" * 60)
model_test = LegacyTransformerModel(nhead, nhid, nlayers, ndropout).to(device)
optimizer_test = optim.SGD(model_test.parameters(), lr=LEARNING_RATE, momentum=0.9)
criterion_test = nn.BCEWithLogitsLoss()
loader_test = DataLoader(dataset_test, batch_size=4, shuffle=True, drop_last=True, collate_fn=clinical_collate_fn)

model_test.train()
batch_t = next(iter(loader_test))
optimizer_test.zero_grad()
age_t = batch_t['age'].to(device)
gender_t = batch_t['gender'].to(device)
lob_t = batch_t['lob'].to(device)
codes_t = batch_t['codes'].to(device)
x_t = torch.cat([age_t.unsqueeze(-1), gender_t.unsqueeze(-1), lob_t.unsqueeze(-1), codes_t], dim=-1)
output_t = model_test(x_t)

targets_flat_t = [item for sublist in batch_t['target'] for item in sublist]
dt_cnt_t = batch_t['dt_cnt'].tolist() if isinstance(batch_t['dt_cnt'], torch.Tensor) else batch_t['dt_cnt']
loss_t = compute_loss_legacy(output_t, targets_flat_t, dt_cnt_t, criterion_test)
loss_t.backward()

# Verify gradients exist
has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model_test.parameters())
assert has_grad, "Model should have non-zero gradients after backward"

torch.nn.utils.clip_grad_norm_(model_test.parameters(), GRADIENT_CLIP)
optimizer_test.step()
print(f"  Training step OK: loss={loss_t.item():.4f}, gradients flow")
print("Training step passed!")
del model_test, optimizer_test, loader_test


TEST 5: Training Step
------------------------------------------------------------
  Training step OK: loss=0.8017, gradients flow
Training step passed!


In [16]:
# TEST 6: Batch Metrics Computation
print("\nTEST 6: Batch Metrics Computation")
print("-" * 60)
model_test = LegacyTransformerModel(nhead, nhid, nlayers, ndropout).to(device)
loader_test = DataLoader(dataset_test, batch_size=4, shuffle=False, drop_last=True, collate_fn=clinical_collate_fn)
batch_t = next(iter(loader_test))

with torch.no_grad():
    age_t = batch_t['age'].to(device)
    gender_t = batch_t['gender'].to(device)
    lob_t = batch_t['lob'].to(device)
    codes_t = batch_t['codes'].to(device)
    x_t = torch.cat([age_t.unsqueeze(-1), gender_t.unsqueeze(-1), lob_t.unsqueeze(-1), codes_t], dim=-1)
    output_t = model_test(x_t)

targets_flat_t = [item for sublist in batch_t['target'] for item in sublist]
dt_cnt_t = batch_t['dt_cnt'].tolist() if isinstance(batch_t['dt_cnt'], torch.Tensor) else batch_t['dt_cnt']
metrics_t = compute_batch_metrics_legacy(output_t, targets_flat_t, dt_cnt_t)

expected_keys = [
    'recall@1', 'recall@5', 'recall@10', 'recall@20', 'recall@50',
    'precision@5', 'precision@10', 'precision@20', 'precision@50',
    'micro_recall@1', 'micro_recall@10', 'micro_recall@20',
    'ndcg@20', 'positive_brier'
]
for key in expected_keys:
    assert key in metrics_t, f"Missing metric: {key}"
    assert isinstance(metrics_t[key], (int, float)), f"Metric {key} should be numeric: {type(metrics_t[key])}"
    assert 0.0 <= metrics_t[key] <= 1.0 or key == 'positive_brier', f"Metric {key} out of range: {metrics_t[key]}"

print(f"  Metrics computed: {len(metrics_t)} keys")
for k, v in metrics_t.items():
    print(f"    {k}: {v:.4f}")
print("Batch metrics passed!")
del model_test, loader_test


TEST 6: Batch Metrics Computation
------------------------------------------------------------
  Metrics computed: 15 keys
    recall@1: 0.0026
    recall@5: 0.0105
    recall@10: 0.0145
    recall@20: 0.0171
    recall@50: 0.0263
    precision@5: 0.0021
    precision@10: 0.0014
    precision@20: 0.0009
    precision@50: 0.0005
    micro_recall@1: 0.0010
    micro_recall@5: 0.0042
    micro_recall@10: 0.0057
    micro_recall@20: 0.0068
    ndcg@20: 0.0041
    positive_brier: 0.2817
Batch metrics passed!


In [17]:
# TEST 7: Evaluation Pipeline (StreamingMetrics)
print("\nTEST 7: Evaluation Pipeline")
print("-" * 60)
model_test = LegacyTransformerModel(nhead, nhid, nlayers, ndropout).to(device)
criterion_test = nn.BCEWithLogitsLoss()
loader_test = DataLoader(dataset_test, batch_size=4, shuffle=False, drop_last=False, collate_fn=clinical_collate_fn)

val_metrics = evaluate(model_test, loader_test, criterion_test, verbose=True)

assert 'val_loss' in val_metrics, "Missing val_loss"
assert 'recall@10' in val_metrics, "Missing recall@10"
assert 'micro_recall@10' in val_metrics, "Missing micro_recall@10"
assert 'ndcg@10' in val_metrics, "Missing ndcg@10"
assert 'mrr' in val_metrics, "Missing mrr"
assert 'positive_brier' in val_metrics, "Missing positive_brier"

print(f"\n  Validation metrics:")
for k, v in val_metrics.items():
    print(f"    {k}: {v:.4f}")
print("Evaluation pipeline passed!")
del model_test, loader_test


TEST 7: Evaluation Pipeline
------------------------------------------------------------
    Val batch 0/5

  Validation metrics:
    val_loss: 0.7862
    mrr: 0.0018
    positive_brier: 0.2911
    recall@1: 0.0000
    micro_recall@1: 0.0000
    precision@1: 0.0000
    ndcg@1: 0.0000
    recall@5: 0.0039
    micro_recall@5: 0.0015
    precision@5: 0.0008
    ndcg@5: 0.0009
    recall@10: 0.0076
    micro_recall@10: 0.0030
    precision@10: 0.0008
    ndcg@10: 0.0014
    recall@20: 0.0076
    micro_recall@20: 0.0030
    precision@20: 0.0004
    ndcg@20: 0.0014
Evaluation pipeline passed!


In [18]:
# TEST 8: Full train_epoch with Metrics
print("\nTEST 8: Full train_epoch with Metrics")
print("-" * 60)
model_test = LegacyTransformerModel(nhead, nhid, nlayers, ndropout).to(device)
optimizer_test = optim.SGD(model_test.parameters(), lr=LEARNING_RATE, momentum=0.9)
criterion_test = nn.BCEWithLogitsLoss()
loader_test = DataLoader(dataset_test, batch_size=4, shuffle=True, drop_last=True, collate_fn=clinical_collate_fn)

loss_tracker_test = LossTracker(window_size=10)
metrics_logger_test = MetricsLogger('test_run', log_dir='/tmp/legacy_test_logs')

epoch_metrics = train_epoch(
    model_test, loader_test, optimizer_test, criterion_test,
    epoch=0, log_interval=1,
    global_step=0,
    loss_tracker=loss_tracker_test,
    metrics_logger=metrics_logger_test,
)

assert 'train_loss' in epoch_metrics, "Missing train_loss"
assert 'aux_loss' in epoch_metrics, "Missing aux_loss (should be 0 for legacy)"
assert epoch_metrics['aux_loss'] == 0.0, "aux_loss should be 0 for legacy"
assert 'train_loss_mean' in epoch_metrics, "Missing loss tracker summary"
assert 'train_loss_std' in epoch_metrics, "Missing loss tracker std"
assert 'global_step' in epoch_metrics, "Missing global_step"

print(f"\n  Epoch metrics:")
for k, v in epoch_metrics.items():
    if isinstance(v, float):
        print(f"    {k}: {v:.4f}")
    else:
        print(f"    {k}: {v}")

# Verify MetricsLogger captured batch metrics
assert len(metrics_logger_test.batch_metrics) > 0, "MetricsLogger should have batch metrics"
metrics_logger_test.save()
assert (Path('/tmp/legacy_test_logs/test_run/batch_metrics.json')).exists(), "batch_metrics.json should exist"
print(f"  MetricsLogger batch entries: {len(metrics_logger_test.batch_metrics)}")
print("Full train_epoch passed!")
del model_test, optimizer_test, loader_test, loss_tracker_test, metrics_logger_test


TEST 8: Full train_epoch with Metrics
------------------------------------------------------------
  Batch 0/5  03/15/26 18:30:36
    Loss: 0.7988 | R@10: 0.007 | R@20: 0.012 | uR@10: 0.003 | P@10: 0.001 | NDCG@20: 0.002 | PosBrier: 0.3000
  Batch 1/5  03/15/26 18:30:37
    Loss: 0.7992 | R@10: 0.008 | R@20: 0.009 | uR@10: 0.003 | P@10: 0.001 | NDCG@20: 0.002 | PosBrier: 0.3022
  Batch 2/5  03/15/26 18:30:37
    Loss: 0.7990 | R@10: 0.013 | R@20: 0.017 | uR@10: 0.005 | P@10: 0.001 | NDCG@20: 0.004 | PosBrier: 0.2994
  Batch 3/5  03/15/26 18:30:38
    Loss: 0.7989 | R@10: 0.008 | R@20: 0.013 | uR@10: 0.003 | P@10: 0.001 | NDCG@20: 0.002 | PosBrier: 0.2973
  Batch 4/5  03/15/26 18:30:39
    Loss: 0.7992 | R@10: 0.004 | R@20: 0.009 | uR@10: 0.002 | P@10: 0.000 | NDCG@20: 0.001 | PosBrier: 0.2973
  Training complete. Average loss: 0.7990

  Epoch metrics:
    train_loss: 0.7990
    aux_loss: 0.0000
    global_step: 5
    num_batches: 5
    train_loss_mean: 0.7990
    train_loss_std: 0.000

In [19]:
# TEST 9: Checkpoint Save/Load
print("\nTEST 9: Checkpoint Save/Load")
print("-" * 60)
import tempfile
model_test = LegacyTransformerModel(nhead, nhid, nlayers, ndropout).to(device)
optimizer_test = optim.SGD(model_test.parameters(), lr=LEARNING_RATE, momentum=0.9)
scheduler_test = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_test, T_max=NUM_EPOCHS)

with tempfile.TemporaryDirectory() as tmpdir:
    ckpt_path = os.path.join(tmpdir, 'test_checkpoint.pt')
    save_checkpoint_local(model_test, optimizer_test, scheduler_test, epoch=3, val_loss=0.5, filepath=ckpt_path)
    assert os.path.exists(ckpt_path), "Checkpoint file should exist"

    model_test2 = LegacyTransformerModel(nhead, nhid, nlayers, ndropout).to(device)
    optimizer_test2 = optim.SGD(model_test2.parameters(), lr=LEARNING_RATE, momentum=0.9)
    scheduler_test2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_test2, T_max=NUM_EPOCHS)
    loaded_epoch, loaded_val_loss = load_checkpoint(ckpt_path, model_test2, optimizer_test2, scheduler_test2)

    assert loaded_epoch == 3, f"Loaded epoch should be 3: {loaded_epoch}"
    assert abs(loaded_val_loss - 0.5) < 1e-6, f"Loaded val_loss should be 0.5: {loaded_val_loss}"

    # Verify weights match
    for (n1, p1), (n2, p2) in zip(model_test.named_parameters(), model_test2.named_parameters()):
        assert torch.allclose(p1, p2), f"Weight mismatch at {n1}"

    print(f"  Checkpoint roundtrip OK: epoch={loaded_epoch}, val_loss={loaded_val_loss}")
print("Checkpoint save/load passed!")
del model_test, model_test2


TEST 9: Checkpoint Save/Load
------------------------------------------------------------
  Checkpoint saved to /var/tmp/tmp9tf_zfck/test_checkpoint.pt
  Checkpoint roundtrip OK: epoch=3, val_loss=0.5
Checkpoint save/load passed!


In [20]:
# TEST 10: LossTracker
print("\nTEST 10: LossTracker")
print("-" * 60)
lt = LossTracker(window_size=5)
for i in range(20):
    lt.log_batch(1.0 - i * 0.04, step=i)
summary = lt.get_epoch_summary()
assert 'train_loss_mean' in summary
assert 'train_loss_std' in summary
assert 'train_loss_improvement' in summary
assert summary['train_loss_improvement'] > 0, "Loss should improve (descending)"
print(f"  Summary: mean={summary['train_loss_mean']:.4f}, std={summary['train_loss_std']:.4f}, "
      f"improvement={summary['train_loss_improvement']:.4f}")
print("LossTracker passed!")

# TEST 11: MetricsLogger Serialization
print("\nTEST 11: MetricsLogger Serialization")
print("-" * 60)
test_data = {
    'np_int': np.int64(42),
    'np_float': np.float32(3.14),
    'np_array': np.array([1, 2, 3]),
    'torch_scalar': torch.tensor(2.71),
    'torch_vector': torch.tensor([1.0, 2.0]),
    'bool_val': np.bool_(True),
}
serialized = MetricsLogger.convert_to_serializable(test_data)
json_str = json.dumps(serialized)
assert isinstance(json_str, str), "Should serialize to JSON string"
print(f"  Serialization OK: {json_str[:80]}...")
print("MetricsLogger serialization passed!")

# TEST 12: StreamingMetrics
print("\nTEST 12: StreamingMetrics")
print("-" * 60)
sm = StreamingMetrics(k_values=(5, 10, 20), vocab_size=100)
fake_preds = torch.randn(8, 100)
fake_targets = [[1, 5, 10], [2, 3], [7], [1, 2, 3, 4], [5], [10, 20], [3, 7], [1]]
sm.update_loss(0.5)
sm.update(fake_preds, fake_targets)
results_sm = sm.compute()
assert 'val_loss' in results_sm
assert 'recall@10' in results_sm
assert 'micro_recall@10' in results_sm
assert 'ndcg@10' in results_sm
assert 'mrr' in results_sm
assert 'positive_brier' in results_sm
print(f"  StreamingMetrics keys: {list(results_sm.keys())}")
print("StreamingMetrics passed!")

print("\n" + "=" * 60)
print("ALL TESTS PASSED!")
print("=" * 60)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


TEST 10: LossTracker
------------------------------------------------------------
  Summary: mean=0.6200, std=0.2307, improvement=0.7600
LossTracker passed!

TEST 11: MetricsLogger Serialization
------------------------------------------------------------
  Serialization OK: {"np_int": 42, "np_float": 3.140000104904175, "np_array": [1, 2, 3], "torch_scal...
MetricsLogger serialization passed!

TEST 12: StreamingMetrics
------------------------------------------------------------
  StreamingMetrics keys: ['val_loss', 'mrr', 'positive_brier', 'recall@5', 'micro_recall@5', 'precision@5', 'ndcg@5', 'recall@10', 'micro_recall@10', 'precision@10', 'ndcg@10', 'recall@20', 'micro_recall@20', 'precision@20', 'ndcg@20']
StreamingMetrics passed!

ALL TESTS PASSED!


## Smoke Test
End-to-end pipeline validation with synthetic data before committing GPU hours.

In [21]:
print("SMOKE TEST: End-to-end pipeline")
print("=" * 60)

dataset_smoke = ClinicalDataset(df_synthetic, target_col='target')
train_size_s = int(0.8 * len(dataset_smoke))
val_size_s = len(dataset_smoke) - train_size_s
train_ds_s, val_ds_s = random_split(dataset_smoke, [train_size_s, val_size_s])

train_loader_s = DataLoader(train_ds_s, batch_size=4, shuffle=True, drop_last=True, collate_fn=clinical_collate_fn)
val_loader_s = DataLoader(val_ds_s, batch_size=4, shuffle=False, collate_fn=clinical_collate_fn)

model_smoke = LegacyTransformerModel(nhead, nhid, nlayers, ndropout).to(device)
optimizer_smoke = optim.SGD(model_smoke.parameters(), lr=LEARNING_RATE, momentum=0.9)
scheduler_smoke = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_smoke, T_max=2)
criterion_smoke = nn.BCEWithLogitsLoss()
loss_tracker_smoke = LossTracker()
metrics_logger_smoke = MetricsLogger('smoke_test', log_dir='/tmp/legacy_smoke_logs')

global_step_s = 0
for ep in range(2):
    print(f"\nSmoke Epoch {ep+1}/2")
    train_metrics_s = train_epoch(
        model_smoke, train_loader_s, optimizer_smoke, criterion_smoke,
        epoch=ep, log_interval=1, global_step=global_step_s,
        loss_tracker=loss_tracker_smoke, metrics_logger=metrics_logger_smoke,
    )
    global_step_s = train_metrics_s['global_step']

    val_metrics_s = evaluate(model_smoke, val_loader_s, criterion_smoke, verbose=True)
    print(f"  Train loss: {train_metrics_s['train_loss']:.4f} | Val loss: {val_metrics_s['val_loss']:.4f}")
    print(f"  Val R@10: {val_metrics_s.get('recall@10', 0):.3f} | Val uR@10: {val_metrics_s.get('micro_recall@10', 0):.3f}")

    epoch_entry = {**train_metrics_s}
    for k, v in val_metrics_s.items():
        epoch_entry[k] = v
    metrics_logger_smoke.log_epoch(ep + 1, epoch_entry)
    scheduler_smoke.step()

metrics_logger_smoke.save()

# Verify log files
smoke_log_path = Path('/tmp/legacy_smoke_logs/smoke_test')
assert (smoke_log_path / 'epoch_metrics.json').exists(), "epoch_metrics.json missing"
assert (smoke_log_path / 'batch_metrics.json').exists(), "batch_metrics.json missing"

with open(smoke_log_path / 'epoch_metrics.json', 'r') as f:
    saved_epochs = json.load(f)
assert len(saved_epochs) == 2, f"Should have 2 epoch entries: {len(saved_epochs)}"

with open(smoke_log_path / 'batch_metrics.json', 'r') as f:
    saved_batches = json.load(f)
assert len(saved_batches) > 0, "Should have batch entries"

print(f"\nSmoke test PASSED!")
print(f"  Epoch metrics entries: {len(saved_epochs)}")
print(f"  Batch metrics entries: {len(saved_batches)}")
print(f"  Keys in epoch entry: {list(saved_epochs[0].keys())[:10]}...")
print(f"  Keys in batch entry: {list(saved_batches[0].keys())[:10]}...")

del model_smoke, optimizer_smoke, scheduler_smoke, dataset_smoke
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SMOKE TEST: End-to-end pipeline
After filtering dt_cnt >= 180: 20 samples
  Pre-processing 0/20...
Pre-processing complete: 20 samples

Smoke Epoch 1/2
  Batch 0/4  03/15/26 18:31:24
    Loss: 0.8006 | R@10: 0.003 | R@20: 0.005 | uR@10: 0.001 | P@10: 0.000 | NDCG@20: 0.001 | PosBrier: 0.2841
  Batch 1/4  03/15/26 18:31:25
    Loss: 0.8007 | R@10: 0.006 | R@20: 0.010 | uR@10: 0.003 | P@10: 0.001 | NDCG@20: 0.002 | PosBrier: 0.2870
  Batch 2/4  03/15/26 18:31:26
    Loss: 0.8006 | R@10: 0.001 | R@20: 0.003 | uR@10: 0.001 | P@10: 0.000 | NDCG@20: 0.000 | PosBrier: 0.2909
  Batch 3/4  03/15/26 18:31:26
    Loss: 0.8003 | R@10: 0.007 | R@20: 0.008 | uR@10: 0.003 | P@10: 0.001 | NDCG@20: 0.002 | PosBrier: 0.2807
  Training complete. Average loss: 0.8006
    Val batch 0/1
  Train loss: 0.8006 | Val loss: 0.7903
  Val R@10: 0.007 | Val uR@10: 0.003

Smoke Epoch 2/2
  Batch 0/4  03/15/26 18:31:27
    Loss: 0.8004 | R@10: 0.003 | R@20: 0.005 | uR@10: 0.001 | P@10: 0.000 | NDCG@20: 0.001 | PosBri

## Main Training Pipeline
Set `EXPERIMENT_ROUND` and data table before running.

In [33]:
# ===========================================================================
# TRAINING CONFIGURATION
# ===========================================================================
# Choose data table:
#   BIGQUERY_TABLE_FULL  = full training set (~15M members)
#   BIGQUERY_TABLE_10PCT = 10% sample (~1.5M members)
TRAINING_TABLE = BIGQUERY_TABLE_10PCT

# Experiment round name (logs to logs/{EXPERIMENT_ROUND}/legacy_replication/)
EXPERIMENT_ROUND = 'exp_round10_legacy'

# Override epochs if needed
NUM_EPOCHS = 1

print(f"Training table: {TRAINING_TABLE}")
print(f"Experiment round: {EXPERIMENT_ROUND}")
print(f"Epochs: {NUM_EPOCHS}")

Training table: edp-prod-storage.edp_ent_sdoheir_cns.a834793_Combined_All_LOB_o3_train_10pct_sample
Experiment round: exp_round10_legacy
Epochs: 1


In [34]:
import time

In [63]:
# ===========================================================================
# 1. LOAD DATA (DO NOT RUN THIS IF HAVING CACHED DATA)
# ===========================================================================
import google.auth
from google.cloud import bigquery
client = bigquery.Client()
start_seconds = time.time()
print("Start loading")
df = load_training_data(table_name=TRAINING_TABLE)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. CREATE DATASET
print("\nCreating dataset...")
dataset = ClinicalDataset(df, target_col=target)

# Compute code frequencies for GradientTierAnalyzer
print("Computing code frequencies for gradient tier analysis...")
code_freq = np.zeros(target_cd_cnt, dtype=np.int64)
for sample in dataset.samples:
    for day_codes in sample['target']:
        for c in day_codes:
            if 0 < c < target_cd_cnt:
                code_freq[c] += 1
print(f"  Non-zero target codes: {(code_freq > 0).sum()} / {target_cd_cnt}")

del df
gc.collect()

# 3. SPLIT TRAIN/VAL
train_size = int((1 - VAL_SPLIT) * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
print(f"Train: {train_size:,} | Val: {val_size:,} | Time spent for preprocessing: {round((time.time() - start_seconds)/60, 3)} min")

Start loading
Loading data from edp-prod-storage.edp_ent_sdoheir_cns.a834793_Combined_All_LOB_o3_train_10pct_sample...
Loaded 1,767,053 rows
After dedup: 1,754,651 unique members

Creating dataset...
After filtering dt_cnt >= 5: 1754651 samples
  Pre-processing 0/1754651...
  Pre-processing 50000/1754651...
  Pre-processing 100000/1754651...
  Pre-processing 150000/1754651...
  Pre-processing 200000/1754651...
  Pre-processing 250000/1754651...
  Pre-processing 300000/1754651...
  Pre-processing 350000/1754651...
  Pre-processing 400000/1754651...
  Pre-processing 450000/1754651...
  Pre-processing 500000/1754651...
  Pre-processing 550000/1754651...
  Pre-processing 600000/1754651...
  Pre-processing 650000/1754651...
  Pre-processing 700000/1754651...
  Pre-processing 750000/1754651...
  Pre-processing 800000/1754651...
  Pre-processing 850000/1754651...
  Pre-processing 900000/1754651...
  Pre-processing 950000/1754651...
  Pre-processing 1000000/1754651...
  Pre-processing 1050000/

In [35]:
train_size

1579185

In [ ]:
# ===========================================================================
# 1-ALT. LOAD DATA WITH LAZY DATASET (USE THIS FOR 11M+ DATA)
# Replaces the eager ClinicalDataset cell above. Stores raw strings,
# parses on-the-fly in __getitem__. Peak RAM: ~30 GB for 11M vs ~1,440 GB eager.
# ===========================================================================
import google.auth
from google.cloud import bigquery
client = bigquery.Client()
start_seconds = time.time()
print("Start loading")
df = load_training_data(table_name=TRAINING_TABLE)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. CREATE LAZY DATASET
print("\nCreating lazy dataset...")
dataset = ClinicalDatasetLazy(df, target_col=target)

# Compute code frequencies via streaming (no materialized targets list)
print("Computing code frequencies for gradient tier analysis...")
code_freq = compute_code_freq_from_strings(dataset.target_strs)

del df
gc.collect()

# 3. SPLIT TRAIN/VAL
train_size = int((1 - VAL_SPLIT) * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
print(f"Train: {train_size:,} | Val: {val_size:,} | Time spent: {round((time.time() - start_seconds)/60, 3)} min")

In [16]:
# ===========================================================================
# SAVE / LOAD PREPROCESSED DATASET
# Streams samples in shards to keep peak memory low.
# codes use int32 (cd_cnt=75516 > int16 max), age/gender int16, lob int8.
# Shard size tuned so each shard's codes array is ~2 GB on disk.
# Save: streams in shards to limit peak memory during write.
# Load: bulk numpy arrays held in a columnar dataset — no per-sample loop.
# ===========================================================================
import pickle

DATASET_CACHE_DIR = 'data/legacy_replication'
SHARD_SIZE = 50_000

class CachedClinicalDataset(Dataset):
    """Drop-in replacement for ClinicalDataset that holds bulk numpy arrays.
    __getitem__ indexes into contiguous arrays and converts to int64 tensors
    via torch.as_tensor().long() — the .long() is the only per-sample copy."""

    def __init__(self, age, gender, lob, codes, dt_cnt, targets, ids):
        self.age = age          # int16 [N, len_dy]
        self.gender = gender    # int16 [N, len_dy]
        self.lob = lob          # int8  [N, len_dy]
        self.codes = codes      # int32 [N, len_dy, len_cd]
        self.dt_cnt = dt_cnt    # int16 [N]
        self.targets = targets  # list of list-of-lists
        self.ids = ids          # list of str

    def __len__(self):
        return len(self.age)

    def __getitem__(self, idx):
        return {
            'age': torch.as_tensor(self.age[idx]).long(),
            'gender': torch.as_tensor(self.gender[idx]).long(),
            'lob': torch.as_tensor(self.lob[idx]).long(),
            'codes': torch.as_tensor(self.codes[idx]).long(),
            'dt_cnt': int(self.dt_cnt[idx]),
            'target': self.targets[idx],
            entity_id: self.ids[idx],
        }


def _to_numpy(val, dtype):
    """Convert tensor or numpy array to numpy with target dtype."""
    if isinstance(val, torch.Tensor):
        return val.numpy().astype(dtype)
    return np.asarray(val, dtype=dtype)


def save_dataset_cache(dataset, train_dataset, val_dataset, code_freq, cache_dir=DATASET_CACHE_DIR):
    """Save preprocessed dataset in shards to limit peak memory during save.
    Works with ClinicalDataset, ClinicalDatasetLazy, and CachedClinicalDataset
    by using __getitem__ instead of accessing internal .samples attribute.
    """
    os.makedirs(cache_dir, exist_ok=True)
    n = len(dataset)
    n_shards = (n + SHARD_SIZE - 1) // SHARD_SIZE
    print(f"Saving {n:,} samples in {n_shards} shards to {cache_dir}/")

    for shard_idx in range(n_shards):
        start = shard_idx * SHARD_SIZE
        end = min(start + SHARD_SIZE, n)
        shard_n = end - start
        print(f"  Shard {shard_idx}/{n_shards}: samples {start:,}-{end-1:,}")

        age_arr = np.empty((shard_n, len_dy), dtype=np.int16)
        gender_arr = np.empty((shard_n, len_dy), dtype=np.int16)
        lob_arr = np.empty((shard_n, len_dy), dtype=np.int8)
        codes_arr = np.empty((shard_n, len_dy, len_cd), dtype=np.int32)
        dt_cnt_arr = np.empty(shard_n, dtype=np.int16)
        targets_shard = []
        ids_shard = []

        for j, i in enumerate(range(start, end)):
            s = dataset[i]
            age_arr[j] = _to_numpy(s['age'], np.int16)
            gender_arr[j] = _to_numpy(s['gender'], np.int16)
            lob_arr[j] = _to_numpy(s['lob'], np.int8)
            codes_arr[j] = _to_numpy(s['codes'], np.int32)
            dt_cnt_arr[j] = s['dt_cnt']
            targets_shard.append(s['target'])
            ids_shard.append(s.get(entity_id))

        np.savez_compressed(
            os.path.join(cache_dir, f'shard_{shard_idx:04d}.npz'),
            age=age_arr, gender=gender_arr, lob=lob_arr,
            codes=codes_arr, dt_cnt=dt_cnt_arr,
        )
        with open(os.path.join(cache_dir, f'shard_{shard_idx:04d}_meta.pkl'), 'wb') as f:
            pickle.dump({'targets': targets_shard, 'ids': ids_shard}, f, protocol=pickle.HIGHEST_PROTOCOL)

        del age_arr, gender_arr, lob_arr, codes_arr, dt_cnt_arr, targets_shard, ids_shard

    gc.collect()

    train_indices = list(train_dataset.indices)
    val_indices = list(val_dataset.indices)
    np.save(os.path.join(cache_dir, 'train_indices.npy'), np.array(train_indices, dtype=np.int32))
    np.save(os.path.join(cache_dir, 'val_indices.npy'), np.array(val_indices, dtype=np.int32))
    np.save(os.path.join(cache_dir, 'code_freq.npy'), code_freq)

    meta = {'n_samples': n, 'n_shards': n_shards, 'shard_size': SHARD_SIZE,
            'len_dy': len_dy, 'len_cd': len_cd,
            'cd_cnt': cd_cnt, 'target_cd_cnt': target_cd_cnt, 'lob_vocab': lob_vocab,
            'train_size': len(train_indices), 'val_size': len(val_indices)}
    with open(os.path.join(cache_dir, 'meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    total_bytes = sum(
        os.path.getsize(os.path.join(cache_dir, f))
        for f in os.listdir(cache_dir)
        if os.path.isfile(os.path.join(cache_dir, f))
    )
    print(f"  Total: {total_bytes / 1024**3:.2f} GB on disk")


def load_dataset_cache(cache_dir=DATASET_CACHE_DIR):
    """Load preprocessed dataset from sharded cache into columnar CachedClinicalDataset.
    No per-sample Python loop — bulk numpy concat only."""
    with open(os.path.join(cache_dir, 'meta.json'), 'r') as f:
        meta = json.load(f)

    n = meta['n_samples']
    n_shards = meta['n_shards']
    print(f"Loading {n:,} samples from {n_shards} shards in {cache_dir}/")

    age_parts, gender_parts, lob_parts, codes_parts, dt_cnt_parts = [], [], [], [], []
    all_targets = []
    all_ids = []

    for shard_idx in range(n_shards):
        print(f"  Shard {shard_idx}/{n_shards}...", end='\r')
        npz = np.load(os.path.join(cache_dir, f'shard_{shard_idx:04d}.npz'))
        age_parts.append(npz['age'])
        gender_parts.append(npz['gender'])
        lob_parts.append(npz['lob'])
        codes_parts.append(npz['codes'])
        dt_cnt_parts.append(npz['dt_cnt'])
        del npz

        with open(os.path.join(cache_dir, f'shard_{shard_idx:04d}_meta.pkl'), 'rb') as f:
            shard_meta = pickle.load(f)
        all_targets.extend(shard_meta['targets'])
        all_ids.extend(shard_meta['ids'])
        del shard_meta

    print(f"\n  Concatenating arrays...")
    dataset = CachedClinicalDataset(
        age=np.concatenate(age_parts),
        gender=np.concatenate(gender_parts),
        lob=np.concatenate(lob_parts),
        codes=np.concatenate(codes_parts),
        dt_cnt=np.concatenate(dt_cnt_parts),
        targets=all_targets,
        ids=all_ids,
    )
    del age_parts, gender_parts, lob_parts, codes_parts, dt_cnt_parts
    gc.collect()

    train_indices = np.load(os.path.join(cache_dir, 'train_indices.npy')).tolist()
    val_indices = np.load(os.path.join(cache_dir, 'val_indices.npy')).tolist()
    train_dataset = torch.utils.data.Subset(dataset, train_indices)
    val_dataset = torch.utils.data.Subset(dataset, val_indices)
    code_freq = np.load(os.path.join(cache_dir, 'code_freq.npy'))

    print(f"  Done: {n:,} samples | Train: {meta['train_size']:,} | Val: {meta['val_size']:,}")
    return dataset, train_dataset, val_dataset, code_freq

print("Dataset save/load functions ready.")

Dataset save/load functions ready.


In [91]:
# Save it for further usage. 
save_dataset_cache(dataset, train_dataset, val_dataset, code_freq)

Saving 1,754,651 samples in 36 shards to data/legacy_replication/
  Shard 0/36: samples 0-49,999
  Shard 1/36: samples 50,000-99,999
  Shard 2/36: samples 100,000-149,999
  Shard 3/36: samples 150,000-199,999
  Shard 4/36: samples 200,000-249,999
  Shard 5/36: samples 250,000-299,999
  Shard 6/36: samples 300,000-349,999
  Shard 7/36: samples 350,000-399,999
  Shard 8/36: samples 400,000-449,999
  Shard 9/36: samples 450,000-499,999
  Shard 10/36: samples 500,000-549,999
  Shard 11/36: samples 550,000-599,999
  Shard 12/36: samples 600,000-649,999
  Shard 13/36: samples 650,000-699,999
  Shard 14/36: samples 700,000-749,999
  Shard 15/36: samples 750,000-799,999
  Shard 16/36: samples 800,000-849,999
  Shard 17/36: samples 850,000-899,999
  Shard 18/36: samples 900,000-949,999
  Shard 19/36: samples 950,000-999,999
  Shard 20/36: samples 1,000,000-1,049,999
  Shard 21/36: samples 1,050,000-1,099,999
  Shard 22/36: samples 1,100,000-1,149,999
  Shard 23/36: samples 1,150,000-1,199,999
 

In [17]:
# --- AFTER KERNEL RESTART: load cached dataset instead of re-running BigQuery ---
dataset, train_dataset, val_dataset, code_freq = load_dataset_cache()
train_size = len(train_dataset)
val_size = len(val_dataset)

Loading 1,754,651 samples from 36 shards in data/legacy_replication/
  Shard 35/36...
  Concatenating arrays...
  Done: 1,754,651 samples | Train: 1,579,185 | Val: 175,466


In [18]:
# 4. CREATE DATALOADERS
train_loader = DataLoader(
    train_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=True,
    num_workers=8, pin_memory=True, prefetch_factor=2,
    persistent_workers=True, drop_last=True, collate_fn=clinical_collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False,
    num_workers=8, pin_memory=True, prefetch_factor=2,
    persistent_workers=True, drop_last=False, collate_fn=clinical_collate_fn
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f"Effective batch size: {MICRO_BATCH_SIZE} x {ACCUMULATION_STEPS} = {MICRO_BATCH_SIZE * ACCUMULATION_STEPS}")


Train batches: 49349 | Val batches: 5484
Effective batch size: 32 x 16 = 512


In [37]:
# ===========================================================================
# 5. CREATE MODEL & SETUP
# ===========================================================================
model = LegacyTransformerModel(nhead, nhid, nlayers, ndropout)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel parameters: {total_params:,}")
if parallel and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Using DataParallel with {torch.cuda.device_count()} GPUs")
model = model.to(device)

scaler = torch.cuda.amp.GradScaler()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.BCEWithLogitsLoss()

# Setup logging (same directory structure as moe_flashattn_4)
if EXPERIMENT_ROUND:
    effective_log_dir = os.path.join('logs', EXPERIMENT_ROUND)
else:
    effective_log_dir = os.path.join('logs', f"legacy_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")

checkpoint_dir = os.path.join(effective_log_dir, EXP_NAME, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

logger = setup_experiment_logging(EXP_NAME, effective_log_dir)
metrics_logger = MetricsLogger(EXP_NAME, effective_log_dir)
loss_tracker = LossTracker(window_size=100)

# GradientTierAnalyzer
gradient_tier_analyzer = GradientTierAnalyzer(
    code_frequencies=code_freq, device=device, log_interval=LOG_INTERVAL
)

# Log config (same format as moe_flashattn_4)
config_dict = {
    'model_type': 'legacy_replication',
    'batch_size': batch_size, 
    'micro_batch_size': MICRO_BATCH_SIZE,
    'accumulation_steps': ACCUMULATION_STEPS,
    'effective_batch_size': MICRO_BATCH_SIZE * ACCUMULATION_STEPS,
    'embedding_size': embedding_size,
    'nhid': nhid, 'nhead': nhead, 'nlayers': nlayers, 'ndropout': ndropout,
    'cd_cnt': cd_cnt, 'target_cd_cnt': target_cd_cnt, 'lob_vocab': lob_vocab,
    'len_dy': len_dy, 'len_cd': len_cd,
    'optimizer': 'SGD', 'learning_rate': LEARNING_RATE, 'momentum': 0.9,
    'scheduler': 'CosineAnnealingLR', 'T_max': NUM_EPOCHS,
    'loss': 'BCEWithLogitsLoss', 'gradient_clip': GRADIENT_CLIP,
    'val_split': VAL_SPLIT, 'num_epochs': NUM_EPOCHS,
    'total_params': total_params,
    'training_table': TRAINING_TABLE,
    'train_samples': train_size, 'val_samples': val_size,
    'parallel': parallel, 'num_gpus': torch.cuda.device_count(),
    'bug_fixes': ['no_log_softmax', 'clip_before_step', 'no_double_update'],
    'mixed_precision': True,
    'differences_from_moe_framework': {
        'no_mixed_precision': False,
        'no_moe': True, 'sgd_not_adamw': True,
        'nhid_512_not_1024': True, 
        'batch_512_not_32': True,
    }
}
metrics_logger.log_config(config_dict)
logger.info(f"Config: {json.dumps(config_dict, indent=2)}")

print(f"\nLog directory: {effective_log_dir}/{EXP_NAME}")
print(f"Checkpoint directory: {checkpoint_dir}")


Model parameters: 24,880,025
Using DataParallel with 4 GPUs


17:59:58 - legacy_replication - INFO - Config: {
  "model_type": "legacy_replication",
  "batch_size": 512,
  "micro_batch_size": 32,
  "accumulation_steps": 16,
  "effective_batch_size": 512,
  "embedding_size": 256,
  "nhid": 512,
  "nhead": 16,
  "nlayers": 6,
  "ndropout": 0.05,
  "cd_cnt": 75516,
  "target_cd_cnt": 6297,
  "lob_vocab": 4,
  "len_dy": 200,
  "len_cd": 80,
  "optimizer": "SGD",
  "learning_rate": 0.01,
  "momentum": 0.9,
  "scheduler": "CosineAnnealingLR",
  "T_max": 1,
  "loss": "BCEWithLogitsLoss",
  "gradient_clip": 0.25,
  "val_split": 0.1,
  "num_epochs": 1,
  "total_params": 24880025,
  "training_table": "edp-prod-storage.edp_ent_sdoheir_cns.a834793_Combined_All_LOB_o3_train_10pct_sample",
  "train_samples": 1579185,
  "val_samples": 175466,
  "parallel": true,
  "num_gpus": 4,
  "bug_fixes": [
    "no_log_softmax",
    "clip_before_step",
    "no_double_update"
  ],
  "mixed_precision": true,
  "differences_from_moe_framework": {
    "no_mixed_precision": fal

  GradientTierAnalyzer initialized:
    Common: 1147 codes
    Medium: 1720 codes
    Rare:   1698 codes
    Tail:   1170 codes

Log directory: logs/exp_round10_legacy/legacy_replication
Checkpoint directory: logs/exp_round10_legacy/legacy_replication/checkpoints


In [ ]:
# AGGRESSIVE CLEANUP: Delete everything from previous failed runs
import gc, torch
from tqdm.notebook import tqdm
# Delete model and training objects if they exist
# for var_name in tqdm(['model', 'optimizer', 'scheduler', 'criterion',
#                  'loss_tracker', 'metrics_logger', 'gradient_tier_analyzer',
#                  'train_metrics', 'val_metrics', 'output', 'x', 'loss']):
#     if var_name in dir():
#         try:
#             exec(f'del {var_name}')
#         except:
#             pass

# gc.collect()
# gc.collect()

if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    
    # Force CUDA to release ALL cached memory
    for gpu_id in range(torch.cuda.device_count()):
        torch.cuda.memory.reset_peak_memory_stats(gpu_id)
        
    gc.collect()
    torch.cuda.empty_cache()
    
    print("After aggressive cleanup:")
    for gpu_id in range(torch.cuda.device_count()):
        total = torch.cuda.get_device_properties(gpu_id).total_memory / 1024**3
        alloc = torch.cuda.memory_allocated(gpu_id) / 1024**3
        reserved = torch.cuda.memory_reserved(gpu_id) / 1024**3
        free = total - reserved
        print(f"  GPU {gpu_id}: allocated={alloc:.2f} GB, reserved={reserved:.2f} GB, free={free:.2f} GB, total={total:.2f} GB")

In [32]:
# ===========================================================================
# 6. TRAINING LOOP
# ===========================================================================
# cleanup_gpu_memory(verbose=True)

print(f"\n{'='*80}")
print(f"Starting training: {NUM_EPOCHS} epochs")
print(f"Optimizer: SGD(lr={LEARNING_RATE}, momentum=0.9)")
print(f"Scheduler: CosineAnnealingLR(T_max={NUM_EPOCHS})")
print(f"Loss: BCEWithLogitsLoss (no pos_weight)")
print(f"Gradient clip: {GRADIENT_CLIP}")
print(f"Micro batch: {MICRO_BATCH_SIZE} | Accum steps: {ACCUMULATION_STEPS} | Effective: {MICRO_BATCH_SIZE * ACCUMULATION_STEPS}")
print(f"{'='*80}\n")

best_val_loss = None
training_history = []
global_step = 0
start_time = time.time()
total_data_load_time = 0.0
total_forward_time = 0.0
total_backward_time = 0.0

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}  |  LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"{'='*60}")

    # Train
    loss_tracker.reset_epoch()
    train_metrics = train_epoch(
        model, train_loader, optimizer, criterion,
        epoch=epoch, log_interval=LOG_INTERVAL,
        global_step=global_step,
        loss_tracker=loss_tracker,
        metrics_logger=metrics_logger,
        logger=logger,
        gradient_tier_analyzer=gradient_tier_analyzer,
        accumulation_steps=ACCUMULATION_STEPS,
        track_gpu_memory=True,
        scaler=scaler,
    )
    global_step = train_metrics['global_step']
    train_loss = train_metrics['train_loss']
    print(f"  Train loss: {train_loss:.6f}")

    total_data_load_time += train_metrics.get('data_load_time_s', 0)
    total_forward_time += train_metrics.get('forward_time_s', 0)
    total_backward_time += train_metrics.get('backward_time_s', 0)

    # Validate
    val_start = time.time()
    val_metrics = evaluate(model, val_loader, criterion, verbose=True, use_amp=True)
    val_time = time.time() - val_start
    val_loss = val_metrics['val_loss']
    print(f"  Val loss:   {val_loss:.6f}")
    print(f"  Val R@10: {val_metrics.get('recall@10', 0):.3f} | "
          f"Val uR@10: {val_metrics.get('micro_recall@10', 0):.3f} | "
          f"Val NDCG@20: {val_metrics.get('ndcg@20', 0):.3f} | "
          f"Val MRR: {val_metrics.get('mrr', 0):.3f}")

    epoch_time = time.time() - epoch_start

    # Combine epoch metrics (includes timing from train_epoch)
    epoch_entry = {
        **train_metrics,
        'lr': optimizer.param_groups[0]['lr'],
        'epoch_time_s': epoch_time,
        'val_time_s': val_time,
        'train_time_s': train_metrics.get('epoch_time_s', 0),
    }
    for k, v in val_metrics.items():
        epoch_entry[k] = v

    training_history.append(epoch_entry)
    metrics_logger.log_epoch(epoch + 1, epoch_entry)

    # Save loss trajectory
    loss_tracker.save_trajectory(
        filepath=os.path.join(effective_log_dir, EXP_NAME, f'loss_trajectory_epoch{epoch}.json')
    )

    # Save best model
    if best_val_loss is None or val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint_local(
            model, optimizer, scheduler, epoch, val_loss,
            os.path.join(checkpoint_dir, 'checkpoint_best.pt')
        )
        # save_checkpoint_gcs(model, optimizer, scheduler, epoch, val_loss, 'best_model.pt')
        print(f"  *** New best model! Val loss: {best_val_loss:.6f}")

    # Save epoch checkpoint
    save_checkpoint_local(
        model, optimizer, scheduler, epoch, val_loss,
        os.path.join(checkpoint_dir, f'checkpoint_epoch{epoch}.pt')
    )
    save_checkpoint_local(
        model, optimizer, scheduler, epoch, val_loss,
        os.path.join(checkpoint_dir, 'checkpoint_latest.pt')
    )

    # Save metrics after each epoch
    metrics_logger.save()
    scheduler.step()

    logger.info(f"Epoch {epoch+1}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}, "
                f"R@10={val_metrics.get('recall@10', 0):.3f}, "
                f"throughput={train_metrics.get('throughput_samples_per_sec', 0):.1f} samples/s, "
                f"time={epoch_time:.0f}s")

total_time = time.time() - start_time

# Compute aggregate training resource metrics
num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
actual_epochs = len(training_history)
num_samples_total = sum(h.get('samples_processed', 0) for h in training_history)
num_tokens_total = num_samples_total * len_dy

time_metrics = compute_training_time_metrics(
    total_train_time=total_time,
    num_epochs=actual_epochs,
    num_samples=num_samples_total,
    num_tokens=num_tokens_total,
    batch_size=MICRO_BATCH_SIZE * ACCUMULATION_STEPS,
    data_load_time=total_data_load_time,
    forward_time=total_forward_time,
    backward_time=total_backward_time,
)

cost_metrics = compute_cost_metrics(
    training_time_sec=total_time,
    num_epochs=actual_epochs,
    gpu_type="T4",
    num_gpus=num_gpus,
)

print(f"\n{'='*80}")
print(f"TRAINING COMPLETE")
print(f"{'='*80}")
print(f"Total time: {total_time:.1f}s ({total_time/3600:.2f}h)")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"\nThroughput:")
print(f"  Samples/sec: {time_metrics['samples_per_sec']:.1f}")
print(f"  Tokens/sec: {time_metrics['tokens_per_sec']:.1f}")
print(f"  Steps/sec: {time_metrics['steps_per_sec']:.2f}")
print(f"  Time/epoch: {time_metrics['time_per_epoch_sec']:.1f}s")
print(f"\nTime Breakdown:")
print(f"  Data loading: {time_metrics.get('data_load_percent', 0):.1f}%")
print(f"  Forward pass: {time_metrics.get('forward_percent', 0):.1f}%")
print(f"  Backward pass: {time_metrics.get('backward_percent', 0):.1f}%")
print(f"\nCost Estimate ({num_gpus}x T4):")
print(f"  This run: ${cost_metrics['cost_usd']:.2f}")
print(f"  Per epoch: ${cost_metrics['cost_per_epoch_usd']:.2f}")
print(f"  Projected 100 epochs: ${cost_metrics.get('projected_cost_100epochs_usd', 0):.2f}")

if torch.cuda.is_available():
    print(f"\nGPU Memory:")
    print(f"  Peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
    print(f"  Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

print(f"{'='*80}")

# Save final results with resource metrics
final_results = {
    'config': config_dict,
    'best_val_loss': best_val_loss,
    'total_time_s': total_time,
    'training_history': training_history,
    'gradient_tier_diagnosis': gradient_tier_analyzer.get_diagnosis() if gradient_tier_analyzer else {},
    'efficiency': time_metrics,
    'cost': cost_metrics,
}
if torch.cuda.is_available():
    final_results['gpu_memory_peak_gib'] = torch.cuda.max_memory_allocated() / 1024**3

metrics_logger.save_final_results(final_results)
metrics_logger.save()

# Print training history
print(f"\nTraining History:")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'R@10':>8} {'uR@10':>8} {'NDCG@20':>8} {'LR':>12} {'Time':>8} {'Samp/s':>8}")
for i, h in enumerate(training_history):
    print(f"{i+1:>6} {h['train_loss']:>12.6f} {h['val_loss']:>12.6f} "
          f"{h.get('recall@10', 0):>8.3f} {h.get('micro_recall@10', 0):>8.3f} "
          f"{h.get('ndcg@20', 0):>8.3f} {h['lr']:>12.8f} "
          f"{h.get('epoch_time_s', 0):>7.0f}s "
          f"{h.get('throughput_samples_per_sec', 0):>8.1f}")


Starting training: 1 epochs
Optimizer: SGD(lr=0.01, momentum=0.9)
Scheduler: CosineAnnealingLR(T_max=1)
Loss: BCEWithLogitsLoss (no pos_weight)
Gradient clip: 0.25
Micro batch: 32 | Accum steps: 16 | Effective: 512


Epoch 1/1  |  LR: 0.010000
  Batch 0/49349  03/16/26 02:21:31

  GPU UTILIZATION CHECK (Batch 0):
   GPU 0: 0.48 GB allocated, 0.57 GB reserved
   GPU 1: 0.02 GB allocated, 0.04 GB reserved
   GPU 2: 0.02 GB allocated, 0.04 GB reserved
   GPU 3: 0.02 GB allocated, 0.04 GB reserved
    Loss: 0.8047 | R@10: 0.013 | R@20: 0.057 | uR@10: 0.002 | P@10: 0.001 | NDCG@20: 0.004 | PosBrier: 0.2708

  GPU tracking for batch 2

GPU MEMORY SUMMARY
GPU   1_after_data_to_gpu 2_after_forward     3_after_backward    
----------------------------------------------------------------------
GPU 0   0.31GB               1.35GB               0.38GB             
GPU 1   0.02GB               1.00GB               0.02GB             
GPU 2   0.02GB               1.00GB               0.02GB        

13:54:36 - legacy_replication - INFO - Epoch 1: train_loss=0.191129, val_loss=0.030352, R@10=0.573, throughput=67.6 samples/s, time=27190s


  Checkpoint saved to logs/legacy_2026-03-16_06-21-09/legacy_replication/checkpoints/checkpoint_latest.pt

TRAINING COMPLETE
Total time: 27191.2s (7.55h)
Best validation loss: 0.030352

Throughput:
  Samples/sec: 58.1
  Tokens/sec: 11615.3
  Steps/sec: 0.11
  Time/epoch: 27191.2s

Time Breakdown:
  Data loading: 0.2%
  Forward pass: 86.7%
  Backward pass: 13.1%

Cost Estimate (4x T4):
  This run: $10.57
  Per epoch: $10.57
  Projected 100 epochs: $1057.43

GPU Memory:
  Peak: 1.69 GiB
  Allocated: 0.39 GiB

Training History:
 Epoch   Train Loss     Val Loss     R@10    uR@10  NDCG@20           LR     Time   Samp/s
     1     0.191129     0.030352    0.573    0.299    0.281   0.01000000   27190s     67.6


### Generate embeddings

In [75]:
# Optional
for name in ['model', 'optimizer', 'scheduler', 
             'train_loader', 'val_loader',
             'training_history', 'gradient_tier_analyzer']:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for gpu_id in range(torch.cuda.device_count()):
        mem_alloc = torch.cuda.memory_allocated(gpu_id) / 1024**3
        mem_reserved = torch.cuda.memory_reserved(gpu_id) / 1024**3
        mem_free = (torch.cuda.get_device_properties(gpu_id).total_memory - torch.cuda.memory_reserved(gpu_id)) / 1024**3
        print(f"GPU {gpu_id}: allocated={mem_alloc:.2f} GiB, reserved={mem_reserved:.2f} GiB, free={mem_free:.2f} GiB")

print(f"\nTraining config:")
print(f"  MICRO_BATCH_SIZE: {MICRO_BATCH_SIZE}")
print(f"  ACCUMULATION_STEPS: {ACCUMULATION_STEPS}")
print(f"  Effective batch size: {MICRO_BATCH_SIZE * ACCUMULATION_STEPS}")
print(f"  Train batches per epoch: {len(train_loader)}")
print(f"  Optimizer steps per epoch: {len(train_loader) // ACCUMULATION_STEPS}")

GPU 0: allocated=0.38 GiB, reserved=0.59 GiB, free=13.98 GiB
GPU 1: allocated=0.03 GiB, reserved=0.17 GiB, free=14.40 GiB
GPU 2: allocated=0.03 GiB, reserved=0.17 GiB, free=14.40 GiB
GPU 3: allocated=0.03 GiB, reserved=0.17 GiB, free=14.40 GiB

Training config:
  MICRO_BATCH_SIZE: 32
  ACCUMULATION_STEPS: 16
  Effective batch size: 512


NameError: name 'train_loader' is not defined

In [76]:
# ===========================================================================
# 7. GENERATE EMBEDDINGS FROM CHECKPOINT & SAVE TO BIGQUERY
#    Multi-GPU: one model replica per GPU, data sharded across GPUs,
#    threads run in parallel.  No DataParallel — each replica has its own
#    forward hook so embedding indexing is always correct.
#
#    Optimizations (adopted from moe_flashattn_3_lob3_downstream):
#    1. Pre-allocated pinned-memory output — zero-copy to numpy
#    2. Per-GPU DataLoader with pin_memory + prefetch (overlap CPU↔GPU)
#    3. torch.inference_mode (faster than no_grad)
#    4. Non-blocking async GPU→CPU via .copy_(non_blocking=True)
#    5. Shared tqdm progress bar with live ETA
# ===========================================================================
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading


class _EmbeddingInferenceDataset(Dataset):
    """Lazy dataset for embedding inference.  Pre-parses lightweight
    columns (age, gender, lob) into compact numpy arrays (~6 GiB for 6.8M
    rows).  The heavy `cd` column (200×80 codes per row = 815 GiB if
    materialized) is kept as raw strings and parsed on-the-fly inside
    __getitem__, which DataLoader workers execute in parallel."""

    def __init__(self, df: pd.DataFrame):
        n = len(df)
        has_lob = 'lob' in df.columns
        print(f"  Pre-parsing {n:,} rows (lightweight columns)...")

        self.ids = df[entity_id].tolist()
        self.dt_cnts = df['dt_cnt'].values.astype(np.int16)
        self.cd_strings = df['cd'].values  # raw strings, parsed lazily

        self.ages = np.zeros((n, len_dy), dtype=np.int16)
        self.genders = np.zeros((n, len_dy), dtype=np.int16)
        self.lobs = np.zeros((n, len_dy), dtype=np.int8)

        age_vals = df['age_in_months'].values
        gender_vals = df['gender_cd'].values
        lob_vals = df['lob'].values if has_lob else None

        for idx in range(n):
            ag = age_vals[idx].split('*')[:len_dy]
            self.ages[idx, :len(ag)] = [min(int(c), 1439) if c else 0 for c in ag]
            gd = gender_vals[idx].split('*')[:len_dy]
            self.genders[idx, :len(gd)] = [min(int(c), 1439) if c else 0 for c in gd]
            if lob_vals is not None:
                self.lobs[idx] = conv_lob(lob_vals[idx], len_dy)
            if idx % 500_000 == 0 and idx > 0:
                print(f"    {idx:,} / {n:,}...")

        mem_gb = (self.ages.nbytes + self.genders.nbytes + self.lobs.nbytes
                  + self.dt_cnts.nbytes) / 1024**3
        print(f"  Pre-parse complete: {mem_gb:.2f} GiB arrays + strings in memory")

    @staticmethod
    def _parse_cd(cd_str):
        """Parse cd string into [len_dy, len_cd] int64 array (called per sample)."""
        codes = np.zeros((len_dy, len_cd), dtype=np.int64)
        days = cd_str.split('*')[:len_dy]
        for d, dy_str in enumerate(days):
            cds = dy_str.split(',')
            for c_idx, c in enumerate(cds[:len_cd]):
                if c:
                    codes[d, c_idx] = int(c)
        return codes

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        codes = self._parse_cd(self.cd_strings[idx])
        return {
            'age': torch.as_tensor(self.ages[idx].astype(np.int64)),
            'gender': torch.as_tensor(self.genders[idx].astype(np.int64)),
            'lob': torch.as_tensor(self.lobs[idx].astype(np.int64)),
            'codes': torch.from_numpy(codes),
            'dt_cnt': int(self.dt_cnts[idx]),
        }


def _collate_emb(batch):
    return {
        'age': torch.stack([b['age'] for b in batch]),
        'gender': torch.stack([b['gender'] for b in batch]),
        'lob': torch.stack([b['lob'] for b in batch]),
        'codes': torch.stack([b['codes'] for b in batch]),
        'dt_cnt': [b['dt_cnt'] for b in batch],
    }


def _extract_on_single_gpu(
    gpu_id: int,
    state_dict: dict,
    dataset: _EmbeddingInferenceDataset,
    embeddings_out: torch.Tensor,
    start_offset: int,
    per_gpu_batch_size: int,
    num_workers: int,
    progress_lock: threading.Lock,
    progress_counter: list,
):
    """Run embedding extraction on one GPU.  Writes directly into the
    pre-allocated pinned-memory tensor at [start_offset : start_offset+n]."""
    gpu_device = torch.device(f'cuda:{gpu_id}')

    replica = LegacyTransformerModel(nhead, nhid, nlayers, ndropout)
    replica.load_state_dict(state_dict)
    replica = replica.to(gpu_device)
    replica.eval()

    activation = {}
    def _hook(module, inp, out):
        activation['enc'] = out.detach()
    handle = replica.transformer_encoder_dy.register_forward_hook(_hook)

    loader = DataLoader(
        dataset,
        batch_size=per_gpu_batch_size,
        shuffle=False,
        collate_fn=_collate_emb,
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=2 if num_workers > 0 else None,
        persistent_workers=num_workers > 0,
        drop_last=False,
    )

    local_idx = start_offset
    with torch.inference_mode():
        for batch in loader:
            bs_actual = batch['age'].shape[0]

            x = torch.cat([
                batch['age'].unsqueeze(-1),
                batch['gender'].unsqueeze(-1),
                batch['lob'].unsqueeze(-1),
                batch['codes'],
            ], dim=-1).to(gpu_device, non_blocking=True)

            dt_cnt_list = batch['dt_cnt']

            with torch.cuda.amp.autocast(enabled=True):
                _ = replica(x)

            enc_out = activation['enc']  # [seq_len, batch, emb_dim]
            patient_embs = torch.stack([
                enc_out[dt_cnt_list[j], j, :] for j in range(bs_actual)
            ]).float()

            embeddings_out[local_idx:local_idx + bs_actual].copy_(
                patient_embs.cpu(), non_blocking=True
            )
            local_idx += bs_actual

            with progress_lock:
                progress_counter[0] += bs_actual

            del x, patient_embs, enc_out
            activation['enc'] = None
            
    torch.cuda.synchronize(gpu_device)
    handle.remove()
    del replica
    torch.cuda.empty_cache()


def generate_and_upload_embeddings(
    checkpoint_path: str,
    data_df: pd.DataFrame,
    per_gpu_batch_size: int = 16,
    num_workers_per_gpu: int = 2,
):
    """
    Multi-GPU embedding generation with pinned-memory pre-allocation,
    DataLoader prefetch, inference_mode, and non-blocking GPU→CPU copy.
    """

    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1

    # --- 0. Free GPU memory ---
    print(f"{'='*80}")
    print(f"EMBEDDING GENERATION PIPELINE  ({num_gpus} GPUs)")
    print(f"{'='*80}")
    if 'model' in globals():
        print("Releasing training model GPU memory...")
        globals()['model'].cpu()
    gc.collect()
    for g in range(num_gpus):
        with torch.cuda.device(g):
            torch.cuda.empty_cache()

    print(f"Checkpoint : {checkpoint_path}")
    print(f"Per-GPU batch: {per_gpu_batch_size}  |  Effective: {per_gpu_batch_size * num_gpus}")
    print(f"Workers/GPU: {num_workers_per_gpu}")

    # --- 1. Load state dict once on CPU ---
    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state_dict = ckpt['model']
    print(f"Checkpoint loaded (epoch {ckpt.get('epoch', '?')}, "
          f"val_loss {ckpt.get('val_loss', '?'):.6f})")
    del ckpt
    
    # --- 3. Pre-parse data into dataset (one-time CPU cost) ---
    print(f"\nPre-parsing data...")
    parse_start = time.time()
    n_samples = len(data_df)
    full_dataset = _EmbeddingInferenceDataset(data_df)
    print(f"Parsed in {time.time() - parse_start:.1f}s")

    # --- 4. Pre-allocate pinned-memory output tensor ---
    embeddings_out = torch.empty(
        (n_samples, embedding_size), dtype=torch.float32, pin_memory=True
    )

    # --- 5. Shard dataset across GPUs & extract in parallel ---
    print(f"\nExtracting embeddings across {num_gpus} GPUs...")
    shard_size = n_samples // num_gpus
    shard_datasets = []
    shard_offsets = []
    for g in range(num_gpus):
        s = g * shard_size
        e = s + shard_size if g < num_gpus - 1 else n_samples
        shard_datasets.append(torch.utils.data.Subset(full_dataset, list(range(s, e))))
        shard_offsets.append(s)
        print(f"  GPU {g}: {e - s:,} rows  (offset {s:,})")

    progress_lock = threading.Lock()
    progress_counter = [0]

    extract_start = time.time()
    pbar = tqdm(total=n_samples, desc=f"Multi-GPU ({num_gpus} GPUs)")

    with ThreadPoolExecutor(max_workers=num_gpus) as executor:
        futures = [
            executor.submit(
                _extract_on_single_gpu,
                gpu_id=g,
                state_dict=state_dict,
                dataset=shard_datasets[g],
                embeddings_out=embeddings_out,
                start_offset=shard_offsets[g],
                per_gpu_batch_size=per_gpu_batch_size,
                num_workers=num_workers_per_gpu,
                progress_lock=progress_lock,
                progress_counter=progress_counter,
            )
            for g in range(num_gpus)
        ]

        last_count = 0
        while not all(f.done() for f in futures):
            with progress_lock:
                current = progress_counter[0]
            if current > last_count:
                pbar.update(current - last_count)
                elapsed_so_far = time.time() - extract_start
                speed = current / elapsed_so_far if elapsed_so_far > 0 else 0
                eta = (n_samples - current) / speed if speed > 0 else 0
                pbar.set_postfix({
                    "speed": f"{speed:,.0f}/s",
                    "ETA": f"{eta:,.0f}s",
                })
            last_count = current
            time.sleep(0.2)

        with progress_lock:
            pbar.update(progress_counter[0] - last_count)
        pbar.close()

        for f in futures:
            f.result()

    elapsed = time.time() - extract_start
    del state_dict
    gc.collect()

    # --- 6. Build DataFrame (zero-copy from pinned memory) ---
    embeddings_np = embeddings_out.numpy()
    embeddings_df = pd.DataFrame(
        embeddings_np, columns=[f'embedding_{i}' for i in range(embedding_size)]
    )
    embeddings_df[entity_id] = full_dataset.ids

    per_stream_speed = n_samples / elapsed if elapsed > 0 else 0
    print(f"\nComplete! Time: {elapsed:.1f}s | Speed: {per_stream_speed:,.0f} samples/s")
    if num_gpus > 1:
        print(f"   Effective: {per_stream_speed * num_gpus:,.0f} samples/s across {num_gpus} GPUs")
    print(f"   Output: {embeddings_np.shape}")

    return embeddings_df

In [77]:
# --- 2. Query data ---
from google.cloud import bigquery 
BQ_SOURCE_TABLE = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_commercial_heldout_transformer_input_4_te_experiment_round_5'
bq_project = GCP_PROJECT
bq_upload_project = 'edp-prod-css-sdoh'
bq_dest_table = BQ_DEST_TABLE
client = bigquery.Client(project=bq_project)
query = f"""
SELECT individual_id, index_dt, age_in_months, gender_cd, cd, dt_cnt, lob
FROM `{BQ_SOURCE_TABLE}`
"""
print(f"\nLoading data from BigQuery...")
df_cm = client.query(query).to_dataframe()
n_samples = len(df_cm)
print(f"Loaded {n_samples:,} rows")
# sample before and after 2023-10-16 (post part will be used for oot validation)
# 0.3 samples for efficent evaluations of embeddings
df_cm['index_dt'] = pd.to_datetime(df_cm['index_dt'])
df_cm_b4_oct = df_cm[df_cm['index_dt'] <= pd.to_datetime("2023-10-16")]
df_cm_after_oct = df_cm[df_cm['index_dt'] > pd.to_datetime("2023-10-16")]
df_cm_b4_oct_sample = df_cm_b4_oct.sample(frac=0.3, random_state=42)
df_cm_after_oct_sample = df_cm_after_oct.sample(frac=0.3, random_state=42)
df_cm_sample = pd.concat([df_cm_b4_oct_sample,
                         df_cm_after_oct])

print(f"Loaded {len(df_cm_sample):,} rows")


Loading data from BigQuery...
Loaded 6,840,066 rows
Loaded 2,886,355 rows


In [78]:
# --- Execute ---
CHECKPOINT_PATH = 'logs/legacy_2026-03-16_06-21-09/legacy_replication/checkpoints/checkpoint_best.pt'
embeddings_df = generate_and_upload_embeddings(
    checkpoint_path=CHECKPOINT_PATH,
    data_df = df_cm_sample
)

EMBEDDING GENERATION PIPELINE  (4 GPUs)
Checkpoint : logs/legacy_2026-03-16_06-21-09/legacy_replication/checkpoints/checkpoint_best.pt
Per-GPU batch: 16  |  Effective: 64
Workers/GPU: 2
Checkpoint loaded (epoch 0, val_loss 0.030352)

Pre-parsing data...
  Pre-parsing 2,886,355 rows (lightweight columns)...
    500,000 / 2,886,355...
    1,000,000 / 2,886,355...
    1,500,000 / 2,886,355...
    2,000,000 / 2,886,355...
    2,500,000 / 2,886,355...
  Pre-parse complete: 2.69 GiB arrays + strings in memory
Parsed in 198.2s

Extracting embeddings across 4 GPUs...
  GPU 0: 721,588 rows  (offset 0)
  GPU 1: 721,588 rows  (offset 721,588)
  GPU 2: 721,588 rows  (offset 1,443,176)
  GPU 3: 721,591 rows  (offset 2,164,764)


Multi-GPU (4 GPUs):   0%|          | 0/2886355 [00:00<?, ?it/s]


Embeddings shape: (2886355, 257)  |  Extract time: 3314.9s  (871 rows/sec)


In [ ]:
df_cm_sample.head()

In [84]:
embeddings_df1 = pd.merge(embeddings_df, 
                         df_cm_sample[['individual_id','index_dt']], 
                         on = 'individual_id', 
                         how = 'inner')

In [86]:
embeddings_df1.drop_duplicates(subset=['individual_id','index_dt'], inplace = True)

In [89]:
# --- 7. Upload to BigQuery ---
BQ_DEST_TABLE = 'edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_exp1_legacy_1_5m_commercial_30pc_sample_embedding'
bq_upload_project = 'edp-prod-css-sdoh'
print(f"\nUploading to {BQ_DEST_TABLE}...")
upload_client = bigquery.Client(project=bq_upload_project)
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True,
)
job = upload_client.load_table_from_dataframe(
    embeddings_df1, BQ_DEST_TABLE, job_config=job_config,
)
job.result()
print(f"Upload complete: {job.output_rows:,} rows written to {BQ_DEST_TABLE}")
print(f"{'='*80}")

for g in range(num_gpus):
    with torch.cuda.device(g):
        torch.cuda.empty_cache()


Uploading to edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_exp1_legacy_1_5m_commercial_30pc_sample_embedding...
Upload complete: 2,862,176 rows written to edp-prod-storage.edp_ent_sdoheir_cns.a964286_te4exp_3lob_exp_round5_exp1_legacy_1_5m_commercial_30pc_sample_embedding


In [54]:
print("embedding done")

embedding done


### Generate Medicare Embeddings (exp_round5 legacy_dbcheck checkpoint)

Generate embeddings for the Medicare population using the legacy model checkpoint.

**Checkpoint:** `logs/exp_round5_3lobs_1-5M_legacy_dbcheck/epoch_2_3/checkpoints/checkpoint_latest.pt`
**Data source:** `edp-prod-storage.edp_ent_sdoheir_cns.a964286_medicare_embedding_raw_features_20240701_20250930`
**Output table:** `edp-prod-storage.edp_ent_sdoheir_cns.a964286_exp_round10_exp2b_medicare_embeddings_20241120_20250930`

Uses the same `generate_and_upload_embeddings` multi-GPU pipeline with exp_round10-style timing output.

In [ ]:
# ===========================================================================
# STEP 1: Clean GPU memory
# ===========================================================================
for name in ['model', 'optimizer', 'scheduler',
             'train_loader', 'val_loader',
             'training_history', 'gradient_tier_analyzer',
             'embeddings_df', 'embeddings_df1']:
    if name in globals():
        del globals()[name]
gc.collect()
if torch.cuda.is_available():
    for gpu_id in range(torch.cuda.device_count()):
        with torch.cuda.device(gpu_id):
            torch.cuda.empty_cache()
        mem_alloc = torch.cuda.memory_allocated(gpu_id) / 1024**3
        mem_free = (torch.cuda.get_device_properties(gpu_id).total_memory - torch.cuda.memory_reserved(gpu_id)) / 1024**3
        print(f"GPU {gpu_id}: allocated={mem_alloc:.2f} GiB, free={mem_free:.2f} GiB")

print("\nGPU memory cleared.")

In [ ]:
# ===========================================================================
# STEP 2: Load Medicare raw features from BigQuery
# ===========================================================================
from google.cloud import bigquery

MEDICARE_RAW_TABLE = (
    'edp-prod-storage.edp_ent_sdoheir_cns'
    '.a964286_medicare_embedding_raw_features_20240701_20250930'
)

bq_client = bigquery.Client(project=GCP_PROJECT)

medicare_sql = f"""
SELECT individual_id, index_dt, age_in_months, gender_cd, cd, dt_cnt, lob
FROM `{MEDICARE_RAW_TABLE}`
"""

print(f"Loading Medicare data from BigQuery...")
print(f"  Source: {MEDICARE_RAW_TABLE}")
df_medicare = bq_client.query(medicare_sql).to_dataframe()
print(f"  Loaded {len(df_medicare):,} rows")
print(f"  Columns: {list(df_medicare.columns)}")
print(f"  dt_cnt range: [{df_medicare['dt_cnt'].min()}, {df_medicare['dt_cnt'].max()}]")
print(f"  index_dt range: [{df_medicare['index_dt'].min()}, {df_medicare['index_dt'].max()}]")
print(f"  Memory: {df_medicare.memory_usage(deep=True).sum() / 1024**3:.2f} GiB")

In [ ]:
# ===========================================================================
# STEP 3: Generate Medicare embeddings from legacy checkpoint
# ===========================================================================

MEDICARE_CHECKPOINT_PATH = (
    'logs/exp_round5_3lobs_1-5M_legacy_dbcheck'
    '/epoch_2_3/checkpoints/checkpoint_latest.pt'
)

INFERENCE_BATCH_SIZE = 16
NUM_WORKERS_PER_GPU = 2

print(f"Checkpoint: {MEDICARE_CHECKPOINT_PATH}")
print(f"Data: {len(df_medicare):,} Medicare members")
print(f"Batch size per GPU: {INFERENCE_BATCH_SIZE}")
print(f"Workers per GPU: {NUM_WORKERS_PER_GPU}")
print()

medicare_embeddings_df = generate_and_upload_embeddings(
    checkpoint_path=MEDICARE_CHECKPOINT_PATH,
    data_df=df_medicare,
    per_gpu_batch_size=INFERENCE_BATCH_SIZE,
    num_workers_per_gpu=NUM_WORKERS_PER_GPU,
)

In [ ]:
# ===========================================================================
# STEP 4: Attach index_dt and verify output
# ===========================================================================

medicare_embeddings_df = pd.merge(
    medicare_embeddings_df,
    df_medicare[['individual_id', 'index_dt']],
    on='individual_id',
    how='inner',
)
medicare_embeddings_df.drop_duplicates(
    subset=['individual_id', 'index_dt'], inplace=True
)

emb_cols = [c for c in medicare_embeddings_df.columns if c.startswith('embedding_')]
print(f"Medicare embeddings ready:")
print(f"  Shape: {medicare_embeddings_df.shape}")
print(f"  Embedding columns: {len(emb_cols)} (embedding_0 .. embedding_{len(emb_cols)-1})")
print(f"  Unique members: {medicare_embeddings_df['individual_id'].nunique():,}")
print(f"  index_dt range: [{medicare_embeddings_df['index_dt'].min()}, "
      f"{medicare_embeddings_df['index_dt'].max()}]")
print(f"\nSample (first 3 rows, first 5 embedding dims):")
display_cols = ['individual_id', 'index_dt'] + emb_cols[:5]
medicare_embeddings_df[display_cols].head(3)

### Continue to train

In [19]:
# ===========================================================================
# 8. CONTINUE TRAINING FROM CHECKPOINT
# ===========================================================================

def continue_training_from_checkpoint(
    checkpoint_path: str,
    train_loader,
    val_loader,
    additional_epochs: int = 5,
    experiment_round: str = None,
    exp_name: str = EXP_NAME,
    early_stopping: EarlyStoppingConfig = None,
    log_dir: str = None,
):
    """
    Resume training from a saved checkpoint with the same optimizer, scheduler,
    and loss configuration. Picks up epoch counter, optimizer state, and
    scheduler state from the checkpoint.

    Args:
        checkpoint_path: path to checkpoint .pt file
        train_loader: DataLoader for training data (must already exist)
        val_loader: DataLoader for validation data (must already exist)
        additional_epochs: how many more epochs to train
        experiment_round: log directory name; defaults to timestamped folder
        exp_name: experiment name subfolder
        early_stopping: optional EarlyStoppingConfig for sub-epoch validation and early stop; None to disable
        log_dir: explicit log directory; when provided, artifacts co-locate with the original run
    """
    # --- 1. Load checkpoint ---
    print(f"\n{'='*80}")
    print("CONTINUE TRAINING FROM CHECKPOINT")
    print(f"{'='*80}")
    print(f"Checkpoint: {checkpoint_path}")
    print(f"Additional epochs: {additional_epochs}")

    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    start_epoch = ckpt['epoch'] + 1
    prev_val_loss = ckpt.get('val_loss', float('inf'))
    ckpt_config = ckpt.get('config', {})
    print(f"Resuming from epoch {start_epoch} (prev val_loss={prev_val_loss:.6f})")

    # --- 2. Rebuild model ---
    cont_model = LegacyTransformerModel(nhead, nhid, nlayers, ndropout)
    cont_model.load_state_dict(ckpt['model'])
    if parallel and torch.cuda.device_count() > 1:
        cont_model = nn.DataParallel(cont_model)
    cont_model = cont_model.to(device)

    # --- 3. Restore optimizer & scheduler ---
    cont_optimizer = optim.SGD(cont_model.parameters(), lr=LEARNING_RATE, momentum=0.9)
    cont_optimizer.load_state_dict(ckpt['optimizer'])

    total_epochs = start_epoch + additional_epochs
    cont_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        cont_optimizer, T_max=total_epochs
    )
    if ckpt.get('scheduler'):
        cont_scheduler.load_state_dict(ckpt['scheduler'])

    cont_criterion = nn.BCEWithLogitsLoss()
    cont_scaler = torch.cuda.amp.GradScaler()

    # --- 4. Setup logging (reuse original log dir when provided) ---
    if log_dir:
        cont_log_dir = log_dir
    elif experiment_round:
        cont_log_dir = os.path.join('logs', experiment_round)
    else:
        cont_log_dir = os.path.join('logs', f"legacy_continued_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}")

    cont_checkpoint_dir = os.path.join(cont_log_dir, exp_name, 'checkpoints')
    os.makedirs(cont_checkpoint_dir, exist_ok=True)

    cont_logger = setup_experiment_logging(exp_name, cont_log_dir, resume=True)
    cont_metrics_logger = MetricsLogger(exp_name, cont_log_dir, resume=True)
    cont_loss_tracker = LossTracker(window_size=100)

    cont_gradient_tier_analyzer = None
    if 'code_freq' in dir() or 'code_freq' in globals():
        cont_gradient_tier_analyzer = GradientTierAnalyzer(
            code_frequencies=code_freq, device=device, log_interval=LOG_INTERVAL
        )

    # --- 4a. Log continued training config ---
    cont_config_dict = {
        'mode': 'continued_training',
        'checkpoint_path': checkpoint_path,
        'start_epoch': start_epoch,
        'additional_epochs': additional_epochs,
        'total_epochs': total_epochs,
        'model_type': 'legacy_replication',
        'batch_size': batch_size,
        'micro_batch_size': MICRO_BATCH_SIZE,
        'accumulation_steps': ACCUMULATION_STEPS,
        'effective_batch_size': MICRO_BATCH_SIZE * ACCUMULATION_STEPS,
        'embedding_size': embedding_size,
        'nhid': nhid, 'nhead': nhead, 'nlayers': nlayers, 'ndropout': ndropout,
        'cd_cnt': cd_cnt, 'target_cd_cnt': target_cd_cnt, 'lob_vocab': lob_vocab,
        'len_dy': len_dy, 'len_cd': len_cd,
        'optimizer': 'SGD', 'learning_rate': LEARNING_RATE, 'momentum': 0.9,
        'scheduler': 'CosineAnnealingLR', 'T_max': total_epochs,
        'loss': 'BCEWithLogitsLoss', 'gradient_clip': GRADIENT_CLIP,
        'parallel': parallel, 'num_gpus': torch.cuda.device_count(),
        'mixed_precision': True,
        'prev_val_loss': prev_val_loss,
    }
    if early_stopping and early_stopping.enabled:
        cont_config_dict['early_stopping'] = {
            'primary_metric': early_stopping.primary_metric,
            'mode': early_stopping.mode,
            'patience': early_stopping.patience,
            'min_delta': early_stopping.min_delta,
            'warmup_steps': early_stopping.warmup_steps,
            'val_check_interval': early_stopping.val_check_interval,
            'val_fraction': early_stopping.val_fraction,
        }
    cont_metrics_logger.log_config(cont_config_dict)
    cont_logger.info(f"Continued Training Config: {json.dumps(cont_config_dict, indent=2)}")

    # --- 4b. Early stopping ---
    es_monitor = None
    max_val_batches = None
    if early_stopping and early_stopping.enabled:
        es_monitor = EarlyStoppingMonitor(config=early_stopping, logger=cont_logger)
        if early_stopping.val_fraction < 1.0:
            total_val_batches = len(val_loader)
            max_val_batches = max(1, int(total_val_batches * early_stopping.val_fraction))
        print(f"\nEarly Stopping Configuration:")
        print(f"  Primary metric: {early_stopping.primary_metric} (mode={early_stopping.mode})")
        print(f"  Patience: {early_stopping.patience} checks")
        print(f"  Warmup: {early_stopping.warmup_steps} optimizer steps")
        print(f"  Validation interval: every {early_stopping.val_check_interval} optimizer steps")
        print(f"  Validation fraction: {early_stopping.val_fraction:.0%} "
              f"({max_val_batches or len(val_loader)} batches)")
        print(f"  Train loss plateau window: {early_stopping.train_loss_slope_window} steps")
        print(f"  Restore best: {early_stopping.restore_best}")

    print(f"Optimizer: SGD(lr={cont_optimizer.param_groups[0]['lr']:.6f}, momentum=0.9)")
    print(f"Scheduler: CosineAnnealingLR(T_max={total_epochs})")
    print(f"Log dir: {cont_log_dir}/{exp_name}")
    print(f"Epochs: {start_epoch} -> {start_epoch + additional_epochs - 1}")
    print(f"{'='*80}\n")

    # --- Sub-epoch validation callback ---
    def _on_optimizer_step(global_step, model, loss_tracker, epoch, batch_idx):
        """Called after every optimizer step. Returns True to stop training."""
        if es_monitor is None:
            return False
        recent = loss_tracker.get_recent_losses(n=100)
        if recent:
            smoothed = sum(recent) / len(recent)
            es_monitor.record_train_loss(global_step, smoothed)
        if not es_monitor.should_validate(global_step):
            return False
        print(f"\n    --- Sub-epoch validation at step {global_step} (batch {batch_idx}) ---")
        val_metrics = evaluate(
            model, val_loader, cont_criterion,
            max_batches=max_val_batches,
            verbose=False,
            use_amp=True,
        )
        status = es_monitor.record_validation(global_step, val_metrics)
        model.train()  # restore training mode (evaluate() leaves model in eval())
        print(f"    {early_stopping.primary_metric}={status['metric_value']:.4f} | "
              f"best={status['best_metric']:.4f}@step{status['best_step']} | "
              f"patience={status['checks_without_improvement']}/{early_stopping.patience} | "
              f"{'WARMUP' if status['in_warmup'] else 'ACTIVE'}")
        if status['improved']:
            best_path = os.path.join(cont_checkpoint_dir, 'checkpoint_best_es.pt')
            save_checkpoint_local(
                model, cont_optimizer, cont_scheduler, epoch, val_metrics.get('val_loss', 0),
                best_path,
            )
            es_monitor.best_checkpoint_path = best_path
            print(f"    *** New best checkpoint saved: {best_path}")
        if es_monitor.detect_train_loss_plateau():
            print(f"    WARNING: Train loss plateau detected (slope < {early_stopping.train_loss_slope_threshold})")
        if status['should_stop']:
            print(f"\n    EARLY STOPPING TRIGGERED at step {global_step}")
            print(f"    Best {early_stopping.primary_metric}: {status['best_metric']:.4f} at step {status['best_step']}")
            return True
        return False

    # --- 5. Training loop ---
    cont_best_val_loss = prev_val_loss
    cont_history = []
    cont_global_step = ckpt.get('config', {}).get('global_step', 0)
    start_time = time.time()
    total_data_load_time = 0.0
    total_forward_time = 0.0
    total_backward_time = 0.0

    for epoch in range(start_epoch, start_epoch + additional_epochs):
        epoch_start = time.time()
        print(f"\n{'='*60}")
        print(f"Epoch {epoch + 1}/{total_epochs}  |  LR: {cont_optimizer.param_groups[0]['lr']:.6f}")
        print(f"{'='*60}")

        cont_loss_tracker.reset_epoch()
        train_metrics = train_epoch(
            cont_model, train_loader, cont_optimizer, cont_criterion,
            epoch=epoch, log_interval=LOG_INTERVAL,
            global_step=cont_global_step,
            loss_tracker=cont_loss_tracker,
            metrics_logger=cont_metrics_logger,
            logger=cont_logger,
            gradient_tier_analyzer=cont_gradient_tier_analyzer,
            accumulation_steps=ACCUMULATION_STEPS,
            track_gpu_memory=False,
            scaler=cont_scaler,
            on_optimizer_step=_on_optimizer_step if es_monitor else None,
        )
        cont_global_step = train_metrics['global_step']
        train_loss = train_metrics['train_loss']
        print(f"  Train loss: {train_loss:.6f}")

        if train_metrics.get('early_stopped', False):
            print(f"\n  Training stopped early at epoch {epoch + 1}, batch {train_metrics.get('stopped_at_batch', '?')}")
            total_data_load_time += train_metrics.get('data_load_time_s', 0)
            total_forward_time += train_metrics.get('forward_time_s', 0)
            total_backward_time += train_metrics.get('backward_time_s', 0)
            val_start = time.time()
            val_metrics = evaluate(cont_model, val_loader, cont_criterion, verbose=True, use_amp=True)
            val_time = time.time() - val_start
            val_loss = val_metrics['val_loss']
            epoch_time = time.time() - epoch_start
            epoch_entry = {**train_metrics, 'lr': cont_optimizer.param_groups[0]['lr'],
                           'epoch_time_s': epoch_time,
                           'val_time_s': val_time,
                           'train_time_s': train_metrics.get('epoch_time_s', 0)}
            for k, v in val_metrics.items():
                epoch_entry[k] = v
            cont_history.append(epoch_entry)
            cont_metrics_logger.log_epoch(epoch + 1, epoch_entry)
            cont_loss_tracker.save_trajectory(
                filepath=os.path.join(cont_log_dir, exp_name, f'loss_trajectory_epoch{epoch}.json')
            )
            cont_metrics_logger.save()
            cont_logger.info(f"Epoch {epoch+1} (EARLY STOPPED): train_loss={train_loss:.6f}, val_loss={val_loss:.6f}, "
                             f"R@10={val_metrics.get('recall@10', 0):.3f}, "
                             f"stopped_at_batch={train_metrics.get('stopped_at_batch', '?')}")
            break

        total_data_load_time += train_metrics.get('data_load_time_s', 0)
        total_forward_time += train_metrics.get('forward_time_s', 0)
        total_backward_time += train_metrics.get('backward_time_s', 0)

        val_start = time.time()
        val_metrics = evaluate(cont_model, val_loader, cont_criterion, verbose=True, use_amp=True)
        val_time = time.time() - val_start
        val_loss = val_metrics['val_loss']
        print(f"  Val loss:   {val_loss:.6f}")
        print(f"  Val R@10: {val_metrics.get('recall@10', 0):.3f} | "
              f"Val uR@10: {val_metrics.get('micro_recall@10', 0):.3f} | "
              f"Val NDCG@20: {val_metrics.get('ndcg@20', 0):.3f} | "
              f"Val MRR: {val_metrics.get('mrr', 0):.3f}")

        epoch_time = time.time() - epoch_start
        epoch_entry = {**train_metrics, 'lr': cont_optimizer.param_groups[0]['lr'],
                       'epoch_time_s': epoch_time,
                       'val_time_s': val_time,
                       'train_time_s': train_metrics.get('epoch_time_s', 0)}
        for k, v in val_metrics.items():
            epoch_entry[k] = v
        cont_history.append(epoch_entry)
        cont_metrics_logger.log_epoch(epoch + 1, epoch_entry)

        cont_loss_tracker.save_trajectory(
            filepath=os.path.join(cont_log_dir, exp_name, f'loss_trajectory_epoch{epoch}.json')
        )

        if cont_best_val_loss is None or val_loss < cont_best_val_loss:
            cont_best_val_loss = val_loss
            save_checkpoint_local(
                cont_model, cont_optimizer, cont_scheduler, epoch, val_loss,
                os.path.join(cont_checkpoint_dir, 'checkpoint_best.pt')
            )
            print(f"  *** New best model! Val loss: {cont_best_val_loss:.6f}")

        save_checkpoint_local(
            cont_model, cont_optimizer, cont_scheduler, epoch, val_loss,
            os.path.join(cont_checkpoint_dir, f'checkpoint_epoch{epoch}.pt')
        )
        save_checkpoint_local(
            cont_model, cont_optimizer, cont_scheduler, epoch, val_loss,
            os.path.join(cont_checkpoint_dir, 'checkpoint_latest.pt')
        )

        cont_metrics_logger.save()
        cont_scheduler.step()

        cont_logger.info(f"Epoch {epoch+1}: train_loss={train_loss:.6f}, val_loss={val_loss:.6f}, "
                         f"R@10={val_metrics.get('recall@10', 0):.3f}, "
                         f"throughput={train_metrics.get('throughput_samples_per_sec', 0):.1f} samples/s, "
                         f"time={epoch_time:.0f}s")

    # --- Restore best checkpoint if early stopped ---
    if es_monitor and es_monitor.should_stop and early_stopping is not None and early_stopping.restore_best:
        best_path = es_monitor.best_checkpoint_path
        if best_path and os.path.exists(best_path):
            print(f"\nRestoring best checkpoint from {best_path}")
            best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
            if parallel and isinstance(cont_model, nn.DataParallel):
                cont_model.module.load_state_dict(best_ckpt['model'])
            else:
                cont_model.load_state_dict(best_ckpt['model'])
            print(f"Restored best model (step {es_monitor.best_step}, "
                  f"{early_stopping.primary_metric}={es_monitor.best_metric:.4f})")

    total_time = time.time() - start_time
    print(f"\n{'='*80}")
    print(f"CONTINUED TRAINING COMPLETE")
    print(f"{'='*80}")
    print(f"Total time: {total_time:.1f}s ({total_time/3600:.2f}h)")
    print(f"Best validation loss: {cont_best_val_loss:.6f}")
    print(f"\nTraining History (continued):")
    print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'R@10':>8} {'uR@10':>8} {'NDCG@20':>8} {'LR':>12}")
    for i, h in enumerate(cont_history):
        ep = start_epoch + i + 1
        print(f"{ep:>6} {h['train_loss']:>12.6f} {h['val_loss']:>12.6f} "
              f"{h.get('recall@10', 0):>8.3f} {h.get('micro_recall@10', 0):>8.3f} "
              f"{h.get('ndcg@20', 0):>8.3f} {h['lr']:>12.8f}")
    print(f"{'='*80}")

    if es_monitor and early_stopping is not None:
        es_summary = es_monitor.get_summary()
        print(f"\nEarly Stopping Summary:")
        print(f"  Stopped early: {es_summary['stopped_early']}")
        print(f"  Total validation checks: {es_summary['total_checks']}")
        print(f"  Best {early_stopping.primary_metric}: {es_summary['best_metric']:.4f} at step {es_summary['best_step']}")
        es_summary_path = os.path.join(cont_log_dir, exp_name, 'early_stopping_summary.json')
        with open(es_summary_path, 'w') as f:
            json.dump(es_summary, f, indent=2)
        print(f"  Summary saved to: {es_summary_path}")

    # --- Save final results (parity with first-round training) ---
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    actual_epochs = len(cont_history)
    num_samples_total = sum(h.get('samples_processed', 0) for h in cont_history)
    num_tokens_total = num_samples_total * len_dy

    time_metrics = compute_training_time_metrics(
        total_train_time=total_time,
        num_epochs=actual_epochs,
        num_samples=num_samples_total,
        num_tokens=num_tokens_total,
        batch_size=MICRO_BATCH_SIZE * ACCUMULATION_STEPS,
        data_load_time=total_data_load_time,
        forward_time=total_forward_time,
        backward_time=total_backward_time,
    )

    cost_metrics = compute_cost_metrics(
        training_time_sec=total_time,
        num_epochs=actual_epochs,
        gpu_type="T4",
        num_gpus=num_gpus,
    )

    cont_final_results = {
        'config': cont_config_dict,
        'best_val_loss': cont_best_val_loss,
        'total_time_s': total_time,
        'training_history': cont_history,
        'gradient_tier_diagnosis': cont_gradient_tier_analyzer.get_diagnosis() if cont_gradient_tier_analyzer else {},
        'efficiency': time_metrics,
        'cost': cost_metrics,
    }
    if torch.cuda.is_available():
        cont_final_results['gpu_memory_peak_gib'] = torch.cuda.max_memory_allocated() / 1024**3
    if es_monitor:
        cont_final_results['early_stopping_summary'] = es_monitor.get_summary()

    cont_metrics_logger.save_final_results(cont_final_results)
    cont_metrics_logger.save()

    return cont_model, cont_optimizer, cont_scheduler, cont_history

In [39]:
# --- Execute: continue training with early stopping ---
CONTINUE_CHECKPOINT = 'logs/legacy_2026-03-16_06-21-09/legacy_replication/checkpoints/checkpoint_best.pt'
ADDITIONAL_EPOCHS = 5

# Reuse the ORIGINAL run's log directory so all artifacts co-locate
# Parent of exp_name/checkpoints/ -> parent of exp_name/ -> log dir
CONTINUE_LOG_DIR = os.path.dirname(os.path.dirname(os.path.dirname(CONTINUE_CHECKPOINT)))

# Early stopping: monitor NDCG@20 every 500 optimizer steps
# with warmup of 1000 steps (protects CosineAnnealingLR exploration phase)
es_config = EarlyStoppingConfig(
    enabled=True,
    primary_metric='ndcg@20',
    mode='max',
    patience=5,
    min_delta=0.001,
    warmup_steps=1000,
    val_check_interval=500,
    val_fraction=0.2,
    train_loss_slope_window=500,
    train_loss_slope_threshold=1e-5,
    restore_best=True,
)

NameError: name 'EarlyStoppingConfig' is not defined

In [ ]:
cont_model, cont_optimizer, cont_scheduler, cont_history = continue_training_from_checkpoint(
    checkpoint_path=CONTINUE_CHECKPOINT,
    train_loader=train_loader,
    val_loader=val_loader,
    additional_epochs=ADDITIONAL_EPOCHS,
    experiment_round=EXPERIMENT_ROUND,
    early_stopping=es_config,
    log_dir=CONTINUE_LOG_DIR,
)

#### Test

In [38]:
# --- Unit tests: early stopping and loss tracking (run after logging + continue-training cells) ---
def test_early_stopping_config():
    """EarlyStoppingConfig: valid creation and __post_init__ validation."""
    c = EarlyStoppingConfig(enabled=True, primary_metric='ndcg@20', mode='max', patience=3)
    assert c.mode == 'max'
    assert c.primary_metric == 'ndcg@20'
    # Invalid mode
    try:
        EarlyStoppingConfig(mode='invalid')
    except AssertionError as e:
        assert "mode" in str(e).lower()
    # Invalid val_fraction
    try:
        EarlyStoppingConfig(val_fraction=1.5)
    except AssertionError as e:
        assert "val_fraction" in str(e).lower()
    print("  EarlyStoppingConfig: OK")

def test_early_stopping_monitor():
    """EarlyStoppingMonitor: should_validate, record_validation, improvement, patience, get_summary."""
    config = EarlyStoppingConfig(patience=2, warmup_steps=0, val_check_interval=100)
    mon = EarlyStoppingMonitor(config=config)
    assert not mon.should_validate(0)
    assert mon.should_validate(100)
    assert mon.should_validate(200)
    # Record improving then non-improving
    s1 = mon.record_validation(100, {'ndcg@20': 0.25})
    assert s1['improved'] and s1['best_metric'] == 0.25 and not s1['should_stop']
    s2 = mon.record_validation(200, {'ndcg@20': 0.30})
    assert s2['improved'] and s2['best_metric'] == 0.30
    s3 = mon.record_validation(300, {'ndcg@20': 0.29})  # worse
    assert not s3['improved'] and s3['checks_without_improvement'] == 1
    s4 = mon.record_validation(400, {'ndcg@20': 0.28})  # still worse
    assert not s4['improved'] and s4['checks_without_improvement'] == 2 and s4['should_stop']
    summary = mon.get_summary()
    assert summary['best_metric'] == 0.30 and summary['stopped_early'] and summary['total_checks'] == 4
    print("  EarlyStoppingMonitor: OK")

def test_loss_tracker_epoch_summary():
    """LossTracker: get_epoch_summary includes train_loss_mean and train_loss_last."""
    lt = LossTracker()
    for i in range(10):
        lt.log_batch(1.0 - i * 0.05, step=i)
    summary = lt.get_epoch_summary()
    assert 'train_loss_mean' in summary
    assert 'train_loss_last' in summary
    assert summary['train_loss_last'] < summary['train_loss_mean']  # decreasing loss
    print("  LossTracker (train_loss_mean / train_loss_last): OK")

def test_continue_training_signature():
    """continue_training_from_checkpoint accepts early_stopping parameter."""
    import inspect
    sig = inspect.signature(continue_training_from_checkpoint)
    assert 'early_stopping' in sig.parameters
    print("  continue_training_from_checkpoint(early_stopping=...): OK")

print("Early stopping & logging unit tests:")
test_early_stopping_config()
test_early_stopping_monitor()
test_loss_tracker_epoch_summary()
test_continue_training_signature()
print("All passed.")

Early stopping & logging unit tests:


NameError: name 'EarlyStoppingConfig' is not defined